# MITONet - Shinnecock Inlet

In [ ]:
import os
import sys
import tensorflow as tf
tf.keras.backend.set_floatx('float64') 

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm, colors as mcolors
from matplotlib.ticker import LinearLocator, ScalarFormatter, FormatStrFormatter
from matplotlib import ticker as mtick
from matplotlib.patches import Patch
import cmocean

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.metrics import mean_absolute_error as mae
import joblib

from IPython.display import display
import matplotlib.ticker as ticker
from matplotlib import rcParams
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import netCDF4 as nc
from tqdm import tqdm

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.patches as mpatches


plt.rc('font', family='serif')
plt.rcParams.update({'font.size': 20,
                     'lines.linewidth': 2,
                     'lines.markersize':10,
                     'axes.labelsize': 16, # fontsize for x and y labels (was 10)
                     'axes.titlesize': 20,
                     'xtick.labelsize': 16,
                     'ytick.labelsize': 16,
                     'legend.fontsize': 16,
                     'axes.linewidth': 2})

import itertools
colors = itertools.cycle(['r','g','b','m','y','c'])
markers = itertools.cycle(['p','d','o','^','s','x',]) #'D','H','v','*'])

from pathlib import Path
try:
    work_dir.exists()
except NameError:
    curr_dir = Path().resolve()
    work_dir = curr_dir.parent  

scripts_dir = work_dir / "src"
sys.path.append(str(scripts_dir.absolute()))

# ae_model_dir = work_dir / "saved_models"
# mito_model_dir = work_dir / "saved_models"
# scalers_dir = work_dir / "scalers"
ae_model_dir = work_dir / "saved_models_mito"
mito_model_dir = work_dir / "saved_models_mito"
scalers_dir = work_dir / "scalers_mito"

#Download and copy your data to this folder
ROOT_DIR = work_dir.parents[1]
SHINNECOCK_DATA_DIR = (
    ROOT_DIR
    / 'data'
    / 'PRJ-5716'
    / 'Simulation--2d-adcirc-simulation-of-tidal-flow-in-shinnecock-inlet-ny-parameterized-by-bottom-friction-coefficient'
    / 'data'
    / 'Model--adcirc-model'
    / 'data'
)
data_dir = SHINNECOCK_DATA_DIR


import mitonet as mito
import autoencoder as ae
import data_loader as dl 
import processing_utils as pu

from importlib import reload as reload

%matplotlib inline

## Specify training configuraiton for scaling purposes and testing cases for visualization

In [ ]:
# Training data for scaling
train_days = [15., 30.]
param_list_train = [0.003, 0.005, 0.01, 0.02, 0.03, 0.04, 0.05]
scaling = True

# Test data
test_days = [5., 60.]
param_list_test = [0.0025, 0.00275, 0.003, 0.005, 0.0075, 0.01, 0.015, 0.02, 0.025, 0.03, 0.04, 0.05, 0.075, 0.1]
# unseen_param_list_test = [0.0025, 0.015, 0.1]
unseen_param_loc = [0,6,-1]


## Load Models

In [ ]:
# Load H Models
ae_model_path_h = str(ae_model_dir) + '/AE_h'
ae_model_h,_ = ae.load_model(ae_model_path_h, comp = False)
mito_model_path_h = str(mito_model_dir) + '/MITO_h'
mito_model_h = tf.keras.models.load_model(mito_model_path_h)

# Load U Models
ae_model_path_u = str(ae_model_dir) + '/AE_u'
ae_model_u,_ = ae.load_model(ae_model_path_u, comp = False)
mito_model_path_u = str(mito_model_dir) + '/MITO_u'
mito_model_u = tf.keras.models.load_model(mito_model_path_u)

# Load V Models
ae_model_path_v = str(ae_model_dir) + '/AE_v'
ae_model_v,_ = ae.load_model(ae_model_path_v, comp = False)
mito_model_path_v = str(mito_model_dir) + '/MITO_v'
mito_model_v = tf.keras.models.load_model(mito_model_path_v)

### Print Model Summary

In [ ]:
# Print AE model summary
ae_model_h.summary()
ae_model_u.summary()
ae_model_v.summary()

In [ ]:
# Print MITONet model summary
mito_model_h.summary()
mito_model_u.summary()
mito_model_v.summary()

## Load and visualize mesh, bathymetry and sensor locations

In [ ]:
# Load mesh
mesh = dl.load_mesh(data_dir/"cf0025")

# Extract fields
field = mesh['bathy']
triangles = mesh['triangles'] - 1  # if your indexing in 'triangles' starts at 1, convert to 0-based
lon = mesh['nodes'][:, 0]
lat = mesh['nodes'][:, 1]

# Sensors (node indices)
sensors = [2775, 2624, 1115]
s1_color = cmocean.cm.phase(0.6)
s2_color = cmocean.cm.phase(1)
s3_color = cmocean.cm.phase(0.8)

# Compute valmin and valmax from valid (unmasked) data
valmax = field[~field.mask].max()
valmin = field[~field.mask].min()

# ---- Single figure with two subplots ----
fig = plt.figure(figsize=(30,20))

# LEFT: full domain
ax = plt.subplot(1, 2, 1, projection=ccrs.PlateCarree())

# Esri World Imagery (satellite) via WMTS
wmts_url = "https://services.arcgisonline.com/arcgis/rest/services/World_Imagery/MapServer/WMTS/1.0.0/WMTSCapabilities.xml"
ax.add_wmts(wmts_url, layer_name="World_Imagery", zorder=0)

# Plot the bathymetry
mesh_plot_left = ax.tripcolor(
    lon, 
    lat, 
    triangles, 
    field, 
    cmap='cmo.deep', 
    edgecolor='k',
    vmin=valmin, 
    vmax=valmax,
    transform=ccrs.PlateCarree()
)

# Desired map extent (lon_min, lon_max, lat_min, lat_max)
extent = [-73.0, -71.95, 40.3, 41.05]
ax.set_extent(extent, crs=ccrs.PlateCarree())

# Customize tick labels
ax.set_xticks([-73, -72.5, -72], crs=ccrs.PlateCarree())
ax.set_yticks([40.4, 40.6, 40.8, 41.0], crs=ccrs.PlateCarree())
lon_formatter = LongitudeFormatter(degree_symbol='°')
lat_formatter = LatitudeFormatter(degree_symbol='°')
ax.xaxis.set_major_formatter(lon_formatter)
ax.yaxis.set_major_formatter(lat_formatter)

# Add sensor markers (noticeable, non-circular)
ax.scatter(lon[sensors[0]], lat[sensors[0]], s=150, marker='o', color=s1_color,   label='Sensor 1', transform=ccrs.PlateCarree(), zorder=3)
ax.scatter(lon[sensors[1]], lat[sensors[1]], s=150, marker='o', color=s2_color, label='Sensor 2', transform=ccrs.PlateCarree(), zorder=3)
ax.scatter(lon[sensors[2]], lat[sensors[2]], s=150, marker='o', color=s3_color,  label='Sensor 3', transform=ccrs.PlateCarree(), zorder=3)
ax.legend(loc='upper left')

# RIGHT: zoomed-in
ax2 = plt.subplot(1, 2, 2, projection=ccrs.PlateCarree())

# Esri World Imagery (satellite) via WMTS
ax2.add_wmts(wmts_url, layer_name="World_Imagery", zorder=0)

# Plot the bathymetry (same vmin/vmax for shared colorbar)
mesh_plot_right = ax2.tripcolor(
    lon, 
    lat, 
    triangles, 
    field, 
    cmap='cmo.deep', 
    edgecolor='k',
    vmin=valmin, 
    vmax=valmax,
    transform=ccrs.PlateCarree()
)

# Desired map extent (lon_min, lon_max, lat_min, lat_max)
extent_zoom = [-72.55, -72.4, 40.8, 40.9]
ax2.set_extent(extent_zoom, crs=ccrs.PlateCarree())

# Customize tick labels
ax2.set_xticks([-72.54, -72.48, -72.4], crs=ccrs.PlateCarree())
ax2.set_yticks([40.8, 40.85, 40.9], crs=ccrs.PlateCarree())
ax2.xaxis.set_major_formatter(lon_formatter)
ax2.yaxis.set_major_formatter(lat_formatter)

# Add sensor markers to zoom panel (same markers/colors)
ax2.scatter(lon[sensors[0]], lat[sensors[0]], s=150, marker='o', color=s1_color,   label='Sensor 1', transform=ccrs.PlateCarree(), zorder=3)
ax2.scatter(lon[sensors[1]], lat[sensors[1]], s=150, marker='o', color=s2_color, label='Sensor 2', transform=ccrs.PlateCarree(), zorder=3)
ax2.scatter(lon[sensors[2]], lat[sensors[2]], s=150, marker='o', color=s3_color,  label='Sensor 3', transform=ccrs.PlateCarree(), zorder=3)

# Single colorbar on the right (attach to both axes)
cbar = fig.colorbar(mesh_plot_right, ax=[ax, ax2], shrink=0.4, pad=0.04)
cbar.set_label('Depth (m)')

plt.show()


## Load Scalers

In [ ]:
scaler_h = joblib.load(str(scalers_dir)+'/ae_scaler_h.save')
scaler_u = joblib.load(str(scalers_dir)+'/ae_scaler_u.save')
scaler_v = joblib.load(str(scalers_dir)+'/ae_scaler_v.save')

t_scaler_h = joblib.load(str(scalers_dir)+'/t_scaler_h.save') 
b_bc_scaler_h = joblib.load(str(scalers_dir)+'/b_bc_scaler_h.save') 
ls_scaler_h = joblib.load(str(scalers_dir)+'/ls_scaler_h.save') 

t_scaler_u = joblib.load(str(scalers_dir)+'/t_scaler_u.save') 
b_bc_scaler_u = joblib.load(str(scalers_dir)+'/b_bc_scaler_u.save') 
ls_scaler_u = joblib.load(str(scalers_dir)+'/ls_scaler_u.save')

t_scaler_v = joblib.load(str(scalers_dir)+'/t_scaler_v.save') 
b_bc_scaler_v = joblib.load(str(scalers_dir)+'/b_bc_scaler_v.save') 
ls_scaler_v = joblib.load(str(scalers_dir)+'/ls_scaler_v.save') 

## Load test data

In [ ]:
Np = len(param_list_test)

mesh = dl.load_mesh(data_dir/"cf0025")
snaps = [np.where(mesh['t']/60/60/24==test_days[0])[0][0], np.where(mesh['t']/60/60/24==test_days[1])[0][0]]
train_snaps = [np.where(mesh['t']/60/60/24==train_days[0])[0][0], np.where(mesh['t']/60/60/24==train_days[1])[0][0]]
Nt_snaps = mesh['t'][snaps[0]:snaps[1]].shape[0]

data_dict = {}
for inx, param in enumerate(param_list_test):
    dirname = 'cf'+str(param).split('.')[1];
    data_dict[inx] = dl.load_variables(data_dir/dirname)

Nn = mesh['nodes'].shape[0]
Ne = mesh['triangles'].shape[0]
Nt = mesh['t'].shape[0]


In [ ]:
dataset_h = np.empty((0, Nn ),)  ## Augment snapshots with parameter value, if needed
dataset_u = np.empty((0, Nn ),)  ## Augment snapshots with parameter value, if needed
dataset_v = np.empty((0, Nn ),)  ## Augment snapshots with parameter value, if needed

key = 'h'
for inx,param in enumerate(param_list_test):
    indx = param_list_test.index(param)
    snap = data_dict[indx][key].data.T    
    dataset_h = np.vstack((dataset_h, snap))

key = 'u'
for inx,param in enumerate(param_list_test):
    indx = param_list_test.index(param)
    snap = data_dict[indx][key].data.T    
    dataset_u = np.vstack((dataset_u, snap))

key = 'v'
for inx,param in enumerate(param_list_test):
    indx = param_list_test.index(param)
    snap = data_dict[indx][key].data.T    
    dataset_v = np.vstack((dataset_v, snap))
    
dataset_u_resh = np.reshape(dataset_u,(Np,Nt,Nn))
dataset_u_crop = dataset_u_resh[:,snaps[0]:snaps[1],:]
dataset_u = np.reshape(dataset_u_crop,(Np*Nt_snaps,Nn))

dataset_h_resh = np.reshape(dataset_h,(Np,Nt,Nn))
dataset_h_crop = dataset_h_resh[:,snaps[0]:snaps[1],:]
dataset_h = np.reshape(dataset_h_crop,(Np*Nt_snaps,Nn))

dataset_v_resh = np.reshape(dataset_v,(Np,Nt,Nn))
dataset_v_crop = dataset_v_resh[:,snaps[0]:snaps[1],:]
dataset_v = np.reshape(dataset_v_crop,(Np*Nt_snaps,Nn))

if scaling:
    data_scaled_h = scaler_h.transform(dataset_h)
    data_scaled_u = scaler_u.transform(dataset_u)
    data_scaled_v = scaler_v.transform(dataset_v)

dataset_r_h = np.reshape(dataset_h,(Np,Nt_snaps,Nn))
dataset_r_u = np.reshape(dataset_u,(Np,Nt_snaps,Nn))
dataset_r_v = np.reshape(dataset_v,(Np,Nt_snaps,Nn))


In [ ]:
data_ls_h = ae_model_h.encoder(data_scaled_h)
latent_dim_h = data_ls_h.shape[1]
data_ls_h = np.array(np.reshape(data_ls_h,(Np,Nt_snaps,latent_dim_h)))

data_ls_u = ae_model_u.encoder(data_scaled_u)
latent_dim_u = data_ls_u.shape[1]
data_ls_u = np.array(np.reshape(data_ls_u,(Np,Nt_snaps,latent_dim_u)))

data_ls_v = ae_model_v.encoder(data_scaled_v)
latent_dim_v = data_ls_v.shape[1]
data_ls_v = np.array(np.reshape(data_ls_v,(Np,Nt_snaps,latent_dim_v)))

In [ ]:
window_size=5
b_par_data_h, b_ic_data_h, b_bc_data_h, t_data_h, target_data_h = pu.latent_multiple_param(param_list_test,data_ls_h,snaps[0],snaps[1],data_dir)
b_par_data_u, b_ic_data_u, b_bc_data_u, t_data_u, target_data_u = pu.latent_multiple_param(param_list_test,data_ls_u,snaps[0],snaps[1],data_dir)
b_par_data_v, b_ic_data_v, b_bc_data_v, t_data_v, target_data_v = pu.latent_multiple_param(param_list_test,data_ls_v,snaps[0],snaps[1],data_dir)

In [ ]:
if scaling is True:
    t_data_h = t_data_h/t_scaler_h
    b_bc_data_h = b_bc_scaler_h.transform(b_bc_data_h)
    b_ic_data_h = ls_scaler_h.transform(b_ic_data_h)
    target_data_h = ls_scaler_h.transform(target_data_h)

    t_data_u = t_data_u/t_scaler_u
    b_bc_data_u = b_bc_scaler_u.transform(b_bc_data_u)
    b_ic_data_u = ls_scaler_u.transform(b_ic_data_u)
    target_data_u = ls_scaler_u.transform(target_data_u)

    t_data_v = t_data_v/t_scaler_v
    b_bc_data_v = b_bc_scaler_v.transform(b_bc_data_v)
    b_ic_data_v = ls_scaler_v.transform(b_ic_data_v)
    target_data_v = ls_scaler_v.transform(target_data_v)

In [ ]:
b_par_data_h = np.reshape(b_par_data_h,(Np,Nt_snaps,1))
b_ic_data_h = np.reshape(b_ic_data_h,(Np,Nt_snaps,latent_dim_h))
b_bc_data_h = np.reshape(b_bc_data_h,(Np,Nt_snaps,75))
target_data_h = np.reshape(target_data_h,(Np,Nt_snaps,latent_dim_h))
t_data_h = np.array((0.20, 0.40, 0.60 , 0.80, 1.))

b_par_data_u = np.reshape(b_par_data_u,(Np,Nt_snaps,1))
b_ic_data_u = np.reshape(b_ic_data_u,(Np,Nt_snaps,latent_dim_u))
b_bc_data_u = np.reshape(b_bc_data_u,(Np,Nt_snaps,75))
target_data_u = np.reshape(target_data_u,(Np,Nt_snaps,latent_dim_u))
t_data_u = np.array((0.20, 0.40, 0.60 , 0.80, 1.))

b_par_data_v = np.reshape(b_par_data_v,(Np,Nt_snaps,1))
b_ic_data_v = np.reshape(b_ic_data_v,(Np,Nt_snaps,latent_dim_v))
b_bc_data_v = np.reshape(b_bc_data_v,(Np,Nt_snaps,75))
target_data_v = np.reshape(target_data_v,(Np,Nt_snaps,latent_dim_v))
t_data_v = np.array((0.20, 0.40, 0.60 , 0.80, 1.))

### Predictions 1

In [ ]:
window_size = 5

In [ ]:
res_h = np.zeros((Np, Nt_snaps, latent_dim_h))

for p in tqdm(range(Np)):
    b_ic = b_ic_data_h[p, 0, :].copy() 
    res_h[p,0,:] = b_ic
    for i in range(0, Nt_snaps, window_size):
        if i+window_size < Nt_snaps:
            par_input = b_par_data_h[p, i+1:i+window_size+1, :] 
            ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
            bc_input = b_bc_data_h[p, i+1:i+window_size+1, :]  
            t_input = np.expand_dims(t_data_h[0:window_size],1)  
    
            prediction = mito_model_h([par_input, ic_input, bc_input, t_input]).numpy()
            res_h[p,i+1:i+window_size+1] = prediction  
            b_ic = prediction[-1].copy()     # Update initial condition for the next iteration
        else:
            diff = Nt_snaps - i - 1
            par_input = b_par_data_h[p, i+1:i+1+diff, :]   
            ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
            bc_input = b_bc_data_h[p, i+1:i+1+diff, :]  
            t_input = np.expand_dims(t_data_h[0:diff],1)  

            prediction = mito_model_h([par_input, ic_input, bc_input, t_input]).numpy()
            res_h[p,i+1:i+diff+1] = prediction  

res_h = ls_scaler_h.inverse_transform(res_h.reshape(Np*Nt_snaps,latent_dim_h))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_h = ae_model_h.decoder(res_h).numpy()
full_res_h = np.squeeze(scaler_h.inverse_transform(full_res_h))

# Reshape data for analysis and visualization
full_res_r_h = np.reshape(full_res_h, (Np, Nt_snaps, Nn))


In [ ]:
res_u = np.zeros((Np, Nt_snaps, latent_dim_u))

for p in tqdm(range(Np)):
    b_ic = b_ic_data_u[p, 0, :].copy() 
    res_u[p,0,:] = b_ic
    for i in range(0, Nt_snaps, window_size):
        if i+window_size < Nt_snaps:
            par_input = b_par_data_u[p, i+1:i+window_size+1, :]   
            ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
            bc_input = b_bc_data_u[p, i+1:i+window_size+1, :]  
            t_input = np.expand_dims(t_data_u[0:window_size],1)  
    
            prediction = mito_model_u([par_input, ic_input, bc_input, t_input]).numpy()
            res_u[p,i+1:i+window_size+1] = prediction  
            b_ic = prediction[-1]   # Update initial condition for the next iteration
        else:
            diff = Nt_snaps - i - 1
            par_input = b_par_data_u[p, i+1:i+1+diff, :]   
            ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
            bc_input = b_bc_data_u[p, i+1:i+1+diff, :]  
            t_input = np.expand_dims(t_data_u[0:diff],1)  

            prediction = mito_model_u([par_input, ic_input, bc_input, t_input]).numpy()
            res_u[p,i+1:i+diff+1] = prediction  
            
res_u = np.squeeze(ls_scaler_u.inverse_transform(res_u.reshape(Np*Nt_snaps,latent_dim_u)))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_u = ae_model_u.decoder(res_u).numpy()
full_res_u = np.squeeze(scaler_u.inverse_transform(full_res_u))

# Reshape data for analysis and visualization
full_res_r_u = np.reshape(full_res_u, (Np, Nt_snaps, Nn))


In [ ]:
res_v = np.zeros((Np, Nt_snaps, latent_dim_v))

for p in tqdm(range(Np)):
    b_ic = b_ic_data_v[p, 0, :].copy() 
    res_v[p,0,:] = b_ic
    for i in range(0, Nt_snaps, window_size):
        if i+window_size < Nt_snaps:
            par_input = b_par_data_v[p, i+1:i+window_size+1, :]   
            ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
            bc_input = b_bc_data_v[p, i+1:i+window_size+1, :]  
            t_input = np.expand_dims(t_data_v[0:window_size],1)  
    
            prediction = mito_model_v([par_input, ic_input, bc_input, t_input]).numpy()
            res_v[p,i+1:i+window_size+1] = prediction  
            b_ic = prediction[-1]    # Update initial condition for the next iteration
        else:
            diff = Nt_snaps - i - 1
            par_input = b_par_data_v[p, i+1:i+1+diff, :]   
            ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
            bc_input = b_bc_data_v[p, i+1:i+1+diff, :]  
            t_input = np.expand_dims(t_data_v[0:diff],1)  

            prediction = mito_model_v([par_input, ic_input, bc_input, t_input]).numpy()
            res_v[p,i+1:i+diff+1] = prediction  
            
res_v = np.squeeze(ls_scaler_v.inverse_transform(res_v.reshape(Np*Nt_snaps,latent_dim_v)))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_v = ae_model_v.decoder(res_v).numpy()
full_res_v = np.squeeze(scaler_v.inverse_transform(full_res_v))

# Reshape data for analysis and visualization
full_res_r_v = np.reshape(full_res_v, (Np, Nt_snaps, Nn))


In [ ]:
np.savez('mito_output.npz', h_data=full_res_r_h, u_data=full_res_r_u, v_data=full_res_r_v)

#### RMSE

In [ ]:
mask_num = 0
rmse_h = np.zeros((Np,Nt_snaps))
rmse_u = np.zeros((Np,Nt_snaps))
rmse_v = np.zeros((Np,Nt_snaps))

ub_s_rmse_h = np.zeros((Np,Nt_snaps))
ub_s_rmse_u = np.zeros((Np,Nt_snaps))
ub_s_rmse_v = np.zeros((Np,Nt_snaps))

max_array_h = np.zeros(Np)
max_array_u = np.zeros(Np)
max_array_v = np.zeros(Np)

n_rmse_h = np.zeros((Np,Nt_snaps))
n_rmse_u = np.zeros((Np,Nt_snaps))
n_rmse_v = np.zeros((Np,Nt_snaps))

for p, param in enumerate(param_list_test[0:]):
    rmse_h[p] = np.sqrt(np.mean((full_res_r_h[p, :, mask_num:] - dataset_r_h[p, :, mask_num:]) ** 2, axis=1))*100
    rmse_u[p] = np.sqrt(np.mean((full_res_r_u[p, :, mask_num:] - dataset_r_u[p, :, mask_num:]) ** 2, axis=1))*100
    rmse_v[p] = np.sqrt(np.mean((full_res_r_v[p, :, mask_num:] - dataset_r_v[p, :, mask_num:]) ** 2, axis=1))*100

    ub_s_rmse_h[p] = np.sqrt(np.mean(((full_res_r_h[p, :, mask_num:]-np.mean(full_res_r_h[p, :, mask_num:],axis=1,keepdims=True))
                                      - (dataset_r_h[p, :, mask_num:]-np.mean(dataset_r_h[p, :, mask_num:],axis=1,keepdims=True))) ** 2, axis=1))*100
    ub_s_rmse_u[p] = np.sqrt(np.mean(((full_res_r_u[p, :, mask_num:]-np.mean(full_res_r_u[p, :, mask_num:],axis=1,keepdims=True))
                                      - (dataset_r_u[p, :, mask_num:]-np.mean(dataset_r_u[p, :, mask_num:],axis=1,keepdims=True))) ** 2, axis=1))*100
    ub_s_rmse_v[p] = np.sqrt(np.mean(((full_res_r_v[p, :, mask_num:]-np.mean(full_res_r_v[p, :, mask_num:],axis=1,keepdims=True))
                                      - (dataset_r_v[p, :, mask_num:]-np.mean(dataset_r_v[p, :, mask_num:],axis=1,keepdims=True))) ** 2, axis=1))*100

    max_array_h[p] = (np.abs(np.max(dataset_r_h[p, :, mask_num:])-np.min(dataset_r_h[p,  :, mask_num:])))*100
    max_array_u[p] = (np.abs(np.max(dataset_r_u[p, :, mask_num:])-np.min(dataset_r_u[p,  :, mask_num:])))*100
    max_array_v[p] = (np.abs(np.max(dataset_r_v[p, :, mask_num:])-np.min(dataset_r_v[p,  :, mask_num:])))*100

    n_rmse_h[p] = rmse_h[p]/max_array_h[p]
    n_rmse_u[p] = rmse_u[p]/max_array_u[p]
    n_rmse_v[p] = rmse_v[p]/max_array_v[p]
    

In [ ]:
mean_rmse_h = np.mean(rmse_h)
mean_rmse_u = np.mean(rmse_u)
mean_rmse_v = np.mean(rmse_v)
print(mean_rmse_h)
print(mean_rmse_u)
print(mean_rmse_v)

In [ ]:
mean_ub_s_rmse_h = np.mean(ub_s_rmse_h)
mean_ub_s_rmse_u = np.mean(ub_s_rmse_u)
mean_ub_s_rmse_v = np.mean(ub_s_rmse_v)
print(mean_ub_s_rmse_h)
print(mean_ub_s_rmse_u)
print(mean_ub_s_rmse_v)

#### Box Plots

##### RMSE

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(15,11), layout='constrained', sharey='row')

# Boxplot for Water Surface Elevation
axs[0].boxplot(rmse_h.T, tick_labels=param_list_test)
axs[0].set_ylabel('Water Surface Elevation (cm)')

# Boxplot for U-Velocity
axs[1].boxplot(rmse_u.T, tick_labels=param_list_test)
axs[1].set_ylabel('U-Velocity  (cm/s)')

# Boxplot for V-Velocity
axs[2].boxplot(rmse_v.T, tick_labels=param_list_test)
axs[2].set_ylabel('V-Velocity (cm/s)')
axs[2].set_xlabel('Roghness Coefficients', fontsize=16)
axs[2].tick_params(axis='x', which='major', rotation=30, labelsize=12)

axs[0].xaxis.set_tick_params(labelbottom=False)
axs[1].xaxis.set_tick_params(labelbottom=False)

# Calculate the min and max values for each dataset
min_h, max_h = np.min(dataset_r_h)*100, np.max(dataset_r_h)*100
min_u, max_u = np.min(dataset_r_u)*100, np.max(dataset_r_u)*100
min_v, max_v = np.min(dataset_r_v)*100, np.max(dataset_r_v)*100

# Add a text box showing the extent for H, U, and V
extent_text_h = f'{min_h:.2f} < H < {max_h:.2f}'
extent_text_u = f'{min_u:.2f} < U < {max_u:.2f}'
extent_text_v = f'{min_v:.2f} < V < {max_v:.2f}'

# Add text box for Water Surface Elevation
axs[0].text(0.16, 0.95, extent_text_h, transform=axs[0].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

# Add text box for U-Velocity
axs[1].text(0.98, 0.95, extent_text_u, transform=axs[1].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

# Add text box for V-Velocity
axs[2].text(0.4, 0.95, extent_text_v, transform=axs[2].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))



##### Unbiased in Space RMSE

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(15,11), layout='constrained', sharey='row')

# Boxplot for Water Surface Elevation
axs[0].boxplot(ub_s_rmse_h.T, tick_labels=param_list_test)
axs[0].set_ylabel('Water Surface Elevation (cm)')

# Boxplot for U-Velocity
axs[1].boxplot(ub_s_rmse_u.T, tick_labels=param_list_test)
axs[1].set_ylabel('U-Velocity  (cm/s)')

# Boxplot for V-Velocity
axs[2].boxplot(ub_s_rmse_v.T, tick_labels=param_list_test)
axs[2].set_ylabel('V-Velocity (cm/s)')
axs[2].set_xlabel('Roghness Coefficients', fontsize=16)
axs[2].tick_params(axis='x', which='major', rotation=30, labelsize=12)

axs[0].xaxis.set_tick_params(labelbottom=False)
axs[1].xaxis.set_tick_params(labelbottom=False)

# Calculate the min and max values for each dataset
min_h, max_h = np.min(dataset_r_h)*100, np.max(dataset_r_h)*100
min_u, max_u = np.min(dataset_r_u)*100, np.max(dataset_r_u)*100
min_v, max_v = np.min(dataset_r_v)*100, np.max(dataset_r_v)*100

# Add a text box showing the extent for H, U, and V
extent_text_h = f'{min_h:.2f} < H < {max_h:.2f}'
extent_text_u = f'{min_u:.2f} < U < {max_u:.2f}'
extent_text_v = f'{min_v:.2f} < V < {max_v:.2f}'

# Add text box for Water Surface Elevation
axs[0].text(0.16, 0.95, extent_text_h, transform=axs[0].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

# Add text box for U-Velocity
axs[1].text(0.98, 0.95, extent_text_u, transform=axs[1].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

# Add text box for V-Velocity
axs[2].text(0.4, 0.95, extent_text_v, transform=axs[2].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))



#### Violin Plots

##### RMSE

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(15,11), layout='constrained', sharey='row')

# violinplot for Water Surface Elevation
h_violin = axs[0].violinplot(rmse_h.T, showmeans=True)
axs[0].set_ylabel('Water Surface Elevation (cm)')

# violinplot for U-Velocity
u_violin = axs[1].violinplot(rmse_u.T, showmeans=True)
axs[1].set_ylabel('U-Velocity  (cm/s)')

# violinplot for V-Velocity
v_violin = axs[2].violinplot(rmse_v.T, showmeans=True)
axs[2].set_ylabel('V-Velocity (cm/s)')
axs[2].set_xlabel('Bottom Friction Coefficients', fontsize=16)
axs[2].set_xticks(np.arange(1, len(param_list_test) + 1))
axs[2].set_xticklabels(param_list_test, fontsize=16)

h_violin['cbars'].set_edgecolor('grey')
h_violin['cmins'].set_edgecolor('grey')
h_violin['cmaxes'].set_edgecolor('grey')
h_violin['cmeans'].set_edgecolor('black')
h_violin['bodies'][0].set_facecolor('orange')
h_violin['bodies'][6].set_facecolor('orange')
h_violin['bodies'][-1].set_facecolor('orange')
h_violin['bodies'][1].set_facecolor('green')
h_violin['bodies'][4].set_facecolor('green')
h_violin['bodies'][8].set_facecolor('green')
h_violin['bodies'][12].set_facecolor('green')

u_violin['cbars'].set_edgecolor('grey')
u_violin['cmins'].set_edgecolor('grey')
u_violin['cmaxes'].set_edgecolor('grey')
u_violin['cmeans'].set_edgecolor('black')
u_violin['bodies'][0].set_facecolor('orange')
u_violin['bodies'][6].set_facecolor('orange')
u_violin['bodies'][-1].set_facecolor('orange')
u_violin['bodies'][1].set_facecolor('green')
u_violin['bodies'][4].set_facecolor('green')
u_violin['bodies'][8].set_facecolor('green')
u_violin['bodies'][12].set_facecolor('green')

v_violin['cbars'].set_edgecolor('grey')
v_violin['cmins'].set_edgecolor('grey')
v_violin['cmaxes'].set_edgecolor('grey')
v_violin['cmeans'].set_edgecolor('black')
v_violin['bodies'][0].set_facecolor('orange')
v_violin['bodies'][6].set_facecolor('orange')
v_violin['bodies'][-1].set_facecolor('orange')
v_violin['bodies'][1].set_facecolor('green')
v_violin['bodies'][4].set_facecolor('green')
v_violin['bodies'][8].set_facecolor('green')
v_violin['bodies'][12].set_facecolor('green')

axs[2].tick_params(axis='x', which='major', rotation=30, labelsize=12)

axs[0].xaxis.set_tick_params(labelbottom=False)
axs[1].xaxis.set_tick_params(labelbottom=False)

# Calculate the min and max values for each dataset
min_h, max_h = np.min(dataset_r_h)*100, np.max(dataset_r_h)*100
min_u, max_u = np.min(dataset_r_u)*100, np.max(dataset_r_u)*100
min_v, max_v = np.min(dataset_r_v)*100, np.max(dataset_r_v)*100

# Add a text violin showing the extent for H, U, and V
extent_text_h = f'{min_h:.2f} < H < {max_h:.2f}'
extent_text_u = f'{min_u:.2f} < U < {max_u:.2f}'
extent_text_v = f'{min_v:.2f} < V < {max_v:.2f}'

# Add text violin for Water Surface Elevation
axs[0].text(0.32, 0.95, extent_text_h, transform=axs[0].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

# Add text violin for U-Velocity
axs[1].text(0.98, 0.95, extent_text_u, transform=axs[1].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

# Add text violin for V-Velocity
axs[2].text(0.4, 0.95, extent_text_v, transform=axs[2].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))


# after creating/colouring your violins…
legend_elems = [
    Patch(facecolor='tab:blue',  alpha=0.33, edgecolor='k', label='Training'),
    Patch(facecolor='tab:green', alpha=0.33, edgecolor='k', label='Validation'),
    Patch(facecolor='tab:orange', alpha=0.33, edgecolor='k', label='Testing'),
]
axs[0].legend(handles=legend_elems, loc='upper left', frameon=True)

##### Unbiased in Space RMSE

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(15,11), layout='constrained', sharey='row')

# violinplot for Water Surface Elevation
axs[0].violinplot(ub_s_rmse_h.T, showmeans=True)
axs[0].set_ylabel('Water Surface Elevation (cm)')

# violinplot for U-Velocity
axs[1].violinplot(ub_s_rmse_u.T, showmeans=True)
axs[1].set_ylabel('U-Velocity  (cm/s)')

# violinplot for V-Velocity
axs[2].violinplot(ub_s_rmse_v.T, showmeans=True)
axs[2].set_ylabel('V-Velocity (cm/s)')
axs[2].set_xlabel('Roghness Coefficients', fontsize=16)
axs[2].set_xticks(np.arange(1, len(param_list_test) + 1))
axs[2].set_xticklabels(param_list_test, fontsize=16)

axs[2].tick_params(axis='x', which='major', rotation=30, labelsize=12)


axs[0].xaxis.set_tick_params(labelbottom=False)
axs[1].xaxis.set_tick_params(labelbottom=False)

# Calculate the min and max values for each dataset
min_h, max_h = np.min(dataset_r_h)*100, np.max(dataset_r_h)*100
min_u, max_u = np.min(dataset_r_u)*100, np.max(dataset_r_u)*100
min_v, max_v = np.min(dataset_r_v)*100, np.max(dataset_r_v)*100

# Add a text violin showing the extent for H, U, and V
extent_text_h = f'{min_h:.2f} < H < {max_h:.2f}'
extent_text_u = f'{min_u:.2f} < U < {max_u:.2f}'
extent_text_v = f'{min_v:.2f} < V < {max_v:.2f}'

# Add text violin for Water Surface Elevation
axs[0].text(0.16, 0.95, extent_text_h, transform=axs[0].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

# Add text violin for U-Velocity
axs[1].text(0.98, 0.95, extent_text_u, transform=axs[1].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

# Add text violin for V-Velocity
axs[2].text(0.4, 0.95, extent_text_v, transform=axs[2].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))



#### Time Series RMSE Plot

In [ ]:
x_train_lower = mesh['t'][train_snaps[0]]/3600/24
x_train_upper = mesh['t'][train_snaps[1]]/3600/24

x_lower = mesh['t'][snaps[0]]/3600/24 
x_upper = mesh['t'][snaps[1]]/3600/24
time = mesh['t'][snaps[0]:snaps[1]]/60/60/24

# colormap=['darkorange', 'darkred', 'darkgreen']

# cmap = plt.get_cmap('Reds')

# # Generate three colors from the colormap
# light_red = cmap(0.3)  # Use 0.3 for a lighter shade
# medium_red = cmap(0.6) # Use 0.6 for a medium shade
# dark_red = cmap(0.9)
# colormap=[light_red,medium_red,dark_red]

# colormap=[(0,0,0), (0.33,0.33,0.33), (0.66,0.66,0.66)]
# colormap=[(0,0,0), (0.25,0.25,0.25), (0.5,0.5,0.5)]
# colormap=[(0.8,0.8,0.8),(0.4,0.4,0.4),(0,0,0)]
colormap = [cmocean.cm.balance(0.375), cmocean.cm.balance(0.25), cmocean.cm.balance(0.125)]

z_order=[2,0,1]

##### RMSE

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(15,11), layout='constrained', sharex=True)

count = 0

for p in [0, 6, 13]:

        axs[1].plot(time,rmse_u[p],label='r: '+str(param_list_test[p]), linewidth=1.5, linestyle='-', color=colormap[count], zorder=z_order[count])
        axs[1].set_xlim(x_lower,x_upper)
        axs[1].axvline(x_train_upper,color='black',linestyle="--")
        axs[1].axvline(x_train_lower,color='black',linestyle="--")

        axs[2].plot(time,rmse_v[p],label='r: '+str(param_list_test[p]), linewidth=1.5, linestyle='-', color=colormap[count], zorder=z_order[count])
        axs[2].set_xlim(x_lower,x_upper)
        axs[2].axvline(x_train_upper,color='black',linestyle="--")
        axs[2].axvline(x_train_lower,color='black',linestyle="--")
        axs[2].set_xlabel('Time (days)')
        
        axs[0].plot(time,rmse_h[p],label='r: '+str(param_list_test[p]), linewidth=1.5, linestyle='-', color=colormap[count], zorder=z_order[count])
        axs[0].set_xlim(x_lower,x_upper)
        axs[0].axvline(x_train_upper,color='black',linestyle="--")
        axs[0].axvline(x_train_lower,color='black',linestyle="--")
        axs[0].legend(bbox_to_anchor=(0.5,1.15), ncol=3, loc='center', title='Bottom Friction Coefficients', title_fontsize=15, fontsize=12.75)

        count += 1

axs[1].axvspan(x_train_lower, x_train_upper, color='darkblue', alpha=0.10, zorder=-1)
axs[2].axvspan(x_train_lower, x_train_upper, color='darkblue', alpha=0.10, zorder=-1)
axs[0].axvspan(x_train_lower, x_train_upper, color='darkblue', alpha=0.10, zorder=-1)


axs[0].set_ylabel('Water Surface Elevation (cm)')

axs[1].set_ylabel('U-Velocity  (cm/s)')

# Boxplot for V-Velocity
axs[2].set_ylabel('V-Velocity (cm/s)')

##### Unbiased Space RMSE

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(15,11), layout='constrained', sharex=True)

count = 0

for p in [0, 6, 13]:

        axs[1].plot(time,ub_s_rmse_u[p],label='r: '+str(param_list_test[p]), linewidth=1.5, linestyle='-', color=colormap[count], zorder=z_order[count])
        axs[1].set_xlim(x_lower,x_upper)
        axs[1].axvline(x_train_upper,color='black',linestyle="--")
        axs[1].axvline(x_train_lower,color='black',linestyle="--")

        axs[2].plot(time,ub_s_rmse_v[p],label='r: '+str(param_list_test[p]), linewidth=1.5, linestyle='-', color=colormap[count], zorder=z_order[count])
        axs[2].set_xlim(x_lower,x_upper)
        axs[2].axvline(x_train_upper,color='black',linestyle="--")
        axs[2].axvline(x_train_lower,color='black',linestyle="--")
        axs[2].set_xlabel('Time (days)')
        
        axs[0].plot(time,ub_s_rmse_h[p],label='r: '+str(param_list_test[p]), linewidth=1.5, linestyle='-', color=colormap[count], zorder=z_order[count])
        axs[0].set_xlim(x_lower,x_upper)
        axs[0].axvline(x_train_upper,color='black',linestyle="--")
        axs[0].axvline(x_train_lower,color='black',linestyle="--")
        axs[0].legend(bbox_to_anchor=(0.5,1.15), ncol=3, loc='center', title='Bottom Friction Coefficients', title_fontsize=15, fontsize=12.75)

        count += 1

axs[1].axvspan(x_train_lower, x_train_upper, color='darkblue', alpha=0.10, zorder=-1)
axs[2].axvspan(x_train_lower, x_train_upper, color='darkblue', alpha=0.10, zorder=-1)
axs[0].axvspan(x_train_lower, x_train_upper, color='darkblue', alpha=0.10, zorder=-1)

axs[0].set_ylabel('Water Surface Elevation (cm)')

axs[1].set_ylabel('U-Velocity  (cm/s)')

# Boxplot for V-Velocity
axs[2].set_ylabel('V-Velocity (cm/s)')

#### Sensor Plots

In [ ]:
param1 = -1
param2 = 0
fig, axs = plt.subplots(3, 3, figsize=(15,10), layout='constrained', gridspec_kw={'width_ratios': [1, 1, 1]})

s1_color = cmocean.cm.phase(0.6)
s2_color = cmocean.cm.phase(1)
s3_color = cmocean.cm.phase(0.8)

axs[0,0].set_ylabel('Water Surface Elevation (cm)')
axs[1,0].set_ylabel('U-Velocity  (cm/s)')
axs[2,0].set_ylabel('V-Velocity (cm/s)')

axs[0,2].set_ylabel('r: 0.1', rotation=270, labelpad=20)
axs[1,2].set_ylabel('r: 0.0025', rotation=270, labelpad=20)
axs[2,2].set_ylabel('r: 0.0025', rotation=270, labelpad=20)
axs[0,2].yaxis.set_label_position("right")
axs[1,2].yaxis.set_label_position("right")
axs[2,2].yaxis.set_label_position("right")


axs[0,0].xaxis.set_tick_params(labelbottom=False)
axs[1,0].xaxis.set_tick_params(labelbottom=False)

axs[0,0].plot(time,full_res_r_h[param1,:,sensors[0]]*100,label='_nolegend_', linewidth=3, linestyle='-',color=s1_color)
first = axs[0,0].plot(time,dataset_r_h[param1,:,sensors[0]]*100,label='ADCIRC', linewidth=2, linestyle='--',color='black')[0]
axs[0,1].plot(time,full_res_r_h[param1,:,sensors[1]]*100,label='_nolegend_', linewidth=3, linestyle='-',color=s2_color)
axs[0,1].plot(time,dataset_r_h[param1,:,sensors[1]]*100,label='_nolegend_', linewidth=2, linestyle='--',color='black')
axs[0,2].plot(time,full_res_r_h[param1,:,sensors[2]]*100,label='_nolegend_', linewidth=3, linestyle='-',color=s3_color)
axs[0,2].plot(time,dataset_r_h[param1,:,sensors[2]]*100,label='_nolegend_', linewidth=2, linestyle='--',color='black')
second = axs[1,0].plot(time,full_res_r_u[param2,:,sensors[0]]*100,label='MITONet (sensor 1)', linewidth=3, linestyle='-',color=s1_color)[0]
axs[1,0].plot(time,dataset_r_u[param2,:,sensors[0]]*100,label='_nolegend_', linewidth=2, linestyle='--',color='black')
third = axs[1,1].plot(time,full_res_r_u[param2,:,sensors[1]]*100,label='MITONet (sensor 2)', linewidth=3, linestyle='-',color=s2_color)[0]
axs[1,1].plot(time,dataset_r_u[param2,:,sensors[1]]*100,label='_nolegend_', linewidth=2, linestyle='--',color='black')
fourth = axs[1,2].plot(time,full_res_r_u[param2,:,sensors[2]]*100,label='MITONet (sensor 3)', linewidth=3, linestyle='-',color=s3_color)[0]
axs[1,2].plot(time,dataset_r_u[param2,:,sensors[2]]*100,label='_nolegend_', linewidth=2, linestyle='--',color='black')
axs[2,0].plot(time,full_res_r_v[param2,:,sensors[0]]*100,label='_nolegend_', linewidth=3, linestyle='-',color=s1_color)
axs[2,0].plot(time,dataset_r_v[param2,:,sensors[0]]*100,label='_nolegend_', linewidth=2, linestyle='--',color='black')
axs[2,1].plot(time,full_res_r_v[param2,:,sensors[1]]*100,label='_nolegend_', linewidth=3, linestyle='-',color=s2_color)
axs[2,1].plot(time,dataset_r_v[param2,:,sensors[1]]*100,label='_nolegend_', linewidth=2, linestyle='--',color='black')
axs[2,2].plot(time,full_res_r_v[param2,:,sensors[2]]*100,label='_nolegend_', linewidth=3, linestyle='-',color=s3_color)
axs[2,2].plot(time,dataset_r_v[param2,:,sensors[2]]*100,label='_nolegend_', linewidth=2, linestyle='--',color='black')
plt.setp(axs[:,0:], xlim=[50,60])

fig.legend(bbox_to_anchor=(0.9, 1.05), handles=[first, second, third, fourth], labels=['ADCIRC', 'MITONet (sensor1)', 'MITONet (sensor2)', 'MITONet (sensor3)'], ncol=4, fontsize=15)

axs[2,0].set_xlabel('Time (days)')
axs[2,1].set_xlabel('Time (days)')
axs[2,2].set_xlabel('Time (days)')

axs[0,0].xaxis.set_tick_params(labelbottom=False)
axs[1,0].xaxis.set_tick_params(labelbottom=False)
axs[0,1].xaxis.set_tick_params(labelbottom=False)
axs[1,1].xaxis.set_tick_params(labelbottom=False)
axs[0,2].xaxis.set_tick_params(labelbottom=False)
axs[1,2].xaxis.set_tick_params(labelbottom=False)

#### Contour Plots

In [ ]:
tstep = -1
param1 = -1
param2 = 0
print(time[tstep])

fig, axs = plt.subplots(4, 3, figsize=(15, 15), layout='constrained')#, sharey='col')

axs[0,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   dataset_r_h[param1,tstep,:]*100, vmin=dataset_r_h[param1,tstep,:].min()*100, 
                   vmax=dataset_r_h[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[0,0].axis('off')
axs[1,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_h[param1,tstep,:]*100, vmin=dataset_r_h[param1,tstep,:].min()*100, 
                   vmax=dataset_r_h[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[1,0].axis('off')
axs[2,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                                      dataset_r_h[param1,tstep,:]*100, vmin=dataset_r_h[param1,tstep,:].min()*100, 
                   vmax=dataset_r_h[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[2,0].axis('off')
axs[2,0].set_xlim([-72.50, -72.46])
axs[2,0].set_ylim([40.82,40.88]) 
last_h = axs[3,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                                      full_res_r_h[param1,tstep,:]*100, vmin=dataset_r_h[param1,tstep,:].min()*100, 
                   vmax=dataset_r_h[param1,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[3,0].axis('off')
axs[3,0].set_xlim([-72.50, -72.46])
axs[3,0].set_ylim([40.82,40.88]) 

axs[0,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   dataset_r_u[param2,tstep,:]*100, vmin=dataset_r_u[param2,tstep,:].min()*100, 
                   vmax=dataset_r_u[param2,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[0,1].axis('off')
axs[1,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_u[param2,tstep,:]*100, vmin=dataset_r_u[param2,tstep,:].min()*100, 
                   vmax=dataset_r_u[param2,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[1,1].axis('off')
axs[2,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                                      dataset_r_u[param2,tstep,:]*100, vmin=dataset_r_u[param2,tstep,:].min()*100, 
                   vmax=dataset_r_u[param2,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[2,1].axis('off')
axs[2,1].set_xlim([-72.50, -72.46])
axs[2,1].set_ylim([40.82,40.88]) 
last_u = axs[3,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                            full_res_r_u[param2,tstep,:]*100, vmin=dataset_r_u[param2,tstep,:].min()*100, 
                            vmax=dataset_r_u[param2,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[3,1].axis('off')
axs[3,1].set_xlim([-72.50, -72.46])
axs[3,1].set_ylim([40.82,40.88]) 

axs[0,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   dataset_r_v[param2,tstep,:]*100, vmin=dataset_r_v[param2,tstep,:].min()*100, 
                   vmax=dataset_r_v[param2,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[0,2].axis('off')
axs[1,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_v[param2,tstep,:]*100, vmin=dataset_r_v[param2,tstep,:].min()*100, 
                   vmax=dataset_r_v[param2,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[1,2].axis('off')
axs[2,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                                      dataset_r_v[param2,tstep,:]*100, vmin=dataset_r_v[param2,tstep,:].min()*100, 
                   vmax=dataset_r_v[param2,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')
axs[2,2].axis('off')
axs[2,2].set_xlim([-72.50, -72.46])
axs[2,2].set_ylim([40.82,40.88]) 
last_v = axs[3,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                            full_res_r_v[param2,tstep,:]*100, vmin=dataset_r_v[param2,tstep,:].min()*100, 
                            vmax=dataset_r_v[param2,tstep,:].max()*100, cmap='cmo.delta', shading='gouraud')

axs[3,2].axis('off')
axs[3,2].set_xlim([-72.50, -72.46])
axs[3,2].set_ylim([40.82,40.88]) 

axs[0,0].set_title('r: 0.1')
axs[0,1].set_title('r: 0.0025')
axs[0,2].set_title('r: 0.0025')

# axs[1,0].set_ylabel('MITONet')
# axs[2,0].set_ylabel('ADCIRC')
# axs[3,0].set_ylabel('MITONet')
axs[0, 0].text(-0.05, 0.5, 'ADCIRC', size=20, ha="center", va="center", 
               rotation=90, transform=axs[0,0].transAxes)
axs[1, 0].text(-0.05, 0.5, 'MITONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[1,0].transAxes)
axs[2, 0].text(-0.05, 0.5, 'ADCIRC', size=20, ha="center", va="center", 
               rotation=90, transform=axs[2,0].transAxes)
axs[3, 0].text(-0.05, 0.5, 'MITONet', size=20, ha="center", va="center", 
               rotation=90, transform=axs[3,0].transAxes)

cbar = plt.colorbar(last_h, orientation='horizontal', aspect=18)
cbar.set_label(label='Water Surface Elevation (cm)', size=20) 
cbar = plt.colorbar(last_u, orientation='horizontal', aspect=18)
cbar.set_label(label='U-Velocity (cm/s)', size=20) 
cbar = plt.colorbar(last_v, orientation='horizontal', aspect=18)
cbar.set_label(label='V-Velocity (cm/s)',  size=20)  

##### MAE over time

In [ ]:
param1 = 0
param2 = -1

# Precompute the two error fields:
err_h  = np.mean(np.abs(dataset_r_h[param2, :, :]  - full_res_r_h[param2, :, :]),axis=0)*100
err_u   = np.mean(np.abs(dataset_r_u[param1, :, :]   - full_res_r_u[param1, :, :]),axis=0)*100
err_v   = np.mean(np.abs(dataset_r_v[param1, :, :]   - full_res_r_v[param1, :, :]),axis=0)*100

fig, axs = plt.subplots(2, 3, figsize=(15,9), layout='constrained')#, sharey='col')
# axs[0,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, full_res_r_h[0,0,:], vmin=-0.5, vmax=0.5, cmap='cmo.delta', shading='gouraud')

last_eh1 = axs[0,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                              err_h, vmin=0, 
                              vmax=err_h.max(), cmap='cmo.amp', shading='gouraud')
axs[0,0].axis('off')

last_eh2 = axs[1,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                              err_h, vmin=0, 
                              vmax=err_h[750:].max(), cmap='cmo.amp', shading='gouraud')
axs[1,0].axis('off')
axs[1,0].set_xlim([-72.50, -72.46])
axs[1,0].set_ylim([40.82,40.88]) 

last_eu1 = axs[0,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                              err_u, vmin=0, 
                              vmax=err_u.max(), cmap='cmo.amp', shading='gouraud')
axs[0,1].axis('off')

last_eu2 = axs[1,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                              err_u, vmin=0, 
                              vmax=err_u[750:].max(), cmap='cmo.amp', shading='gouraud')
axs[1,1].axis('off')
axs[1,1].set_xlim([-72.50, -72.46])
axs[1,1].set_ylim([40.82,40.88]) 

last_ev1 = axs[0,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                              err_v, vmin=0, 
                              vmax=err_v.max(), cmap='cmo.amp', shading='gouraud')
axs[0,2].axis('off')

last_ev2 = axs[1,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                              err_v, vmin=0, 
                              vmax=err_v[750:].max(), cmap='cmo.amp', shading='gouraud')
axs[1,2].axis('off')
axs[1,2].set_xlim([-72.50, -72.46])
axs[1,2].set_ylim([40.82,40.88]) 

axs[0, 0].text(-0.05, 0.5, 'Full Domain', size=20, ha="center", va="center", 
               rotation=90, transform=axs[0,0].transAxes)
axs[1, 0].text(-0.05, 0.5, 'Zoomed-in', size=20, ha="center", va="center", 
               rotation=90, transform=axs[1,0].transAxes)

axs[0, 0].set_title('Water Surface Elevation (r: 0.1)', size=20,transform=axs[0,0].transAxes)
axs[0, 1].set_title('U-Velocity (r: 0.0025)', size=20,transform=axs[0,1].transAxes)
axs[0, 2].set_title('V-Velocity (r: 0.0025)', size=20,transform=axs[0,2].transAxes)

cbar = plt.colorbar(last_eh1, orientation='horizontal', aspect=18)
# cbar.set_label(label='Mean Abs. Error (cm)', size=20) 
cbar = plt.colorbar(last_eu1, orientation='horizontal', aspect=18)
# cbar.set_label(label='Mean Abs. Error (cm/s)', size=20) 
cbar = plt.colorbar(last_ev1, orientation='horizontal', aspect=18)
# cbar.set_label(label='Mean Abs. Error (cm/s)',  size=20) 

cbar = plt.colorbar(last_eh2, orientation='horizontal', aspect=18)
cbar.set_label(label='Mean Abs. Error (cm)', size=20) 
cbar = plt.colorbar(last_eu2, orientation='horizontal', aspect=18)
cbar.set_label(label='Mean Abs. Error (cm/s)', size=20) 
cbar = plt.colorbar(last_ev2, orientation='horizontal', aspect=18)
cbar.set_label(label='Mean Abs. Error (cm/s)',  size=20) 

##### MAE and Norm over time

In [ ]:
param1 = 0
param2 = 2
param3 = 3
param4 = 6

# Precompute the two error fields:
err_u1   = np.mean(np.abs(dataset_r_u[param1, :, :]   - full_res_r_u[param1, :, :]),axis=0)*100
err_u2   = np.mean(np.abs(dataset_r_u[param2, :, :]   - full_res_r_u[param2, :, :]),axis=0)*100
err_u3   = np.mean(np.abs(dataset_r_u[param3, :, :]   - full_res_r_u[param3, :, :]),axis=0)*100
err_u4   = np.mean(np.abs(dataset_r_u[param4, :, :]   - full_res_r_u[param4, :, :]),axis=0)*100

fig, axs = plt.subplots(1, 4, figsize=(10,2), layout='constrained')#, sharey='col')

fig.supylabel('U-Velocity',fontsize=12)


eu1 = axs[0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                              err_u1, vmin=0, 
                              vmax=20, cmap='cmo.amp', shading='gouraud')
axs[0].axis('off')
axs[0].set_ylabel('MAE', size=20,transform=axs[1].transAxes)
axs[0].set_title('r: '+str(param_list_test[param1]), size=12)


eu2 = axs[1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                              err_u2, vmin=0, 
                              vmax=20, cmap='cmo.amp', shading='gouraud')
axs[1].axis('off')
axs[1].set_title('r: '+str(param_list_test[param2]), size=12)

eu3 = axs[2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                              err_u3, vmin=0, 
                              vmax=20, cmap='cmo.amp', shading='gouraud')
axs[2].axis('off')
axs[2].set_title('r: '+str(param_list_test[param3]), size=12)


eu4 = axs[3].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                              err_u4, vmin=0, 
                              vmax=20, cmap='cmo.amp', shading='gouraud')
axs[3].axis('off')
axs[3].set_title('r: '+str(param_list_test[param4]), size=12)

cbar = plt.colorbar(eu4, orientation='vertical', aspect=18)
cbar.set_label(label='Mean Abs. Error (cm/s)', size=12) 
cbar.ax.tick_params(labelsize=12) 

### Bias

In [ ]:
# shapes
P, T, S = full_res_r_h.shape  
param_arr = np.asarray(param_list_test, dtype=float)  

norm = mcolors.Normalize(vmin=np.nanmin(param_arr), vmax=np.nanmax(param_arr))
cmap = 'cmo.haline'

c_all = np.repeat(param_arr, T)

fig, axs = plt.subplots(3, 3, figsize=(12,10), layout='constrained')

axs[2,0].set_xlabel('True')
axs[2,1].set_xlabel('True')
axs[2,2].set_xlabel('True')

axs[0,0].set_ylabel('Predicted')
axs[1,0].set_ylabel('Predicted')
axs[2,0].set_ylabel('Predicted')

axs[0,2].set_ylabel('Water Surface Elevation (cm)')
axs[1,2].set_ylabel('U-Velocity  (cm/s)')
axs[2,2].set_ylabel('V-Velocity (cm/s)')

axs[0,2].yaxis.set_label_position("right")
axs[1,2].yaxis.set_label_position("right")
axs[2,2].yaxis.set_label_position("right")

# axs[0,0].xaxis.set_tick_params(labelbottom=False)
# axs[1,0].xaxis.set_tick_params(labelbottom=False)
# axs[0,1].xaxis.set_tick_params(labelbottom=False)
# axs[1,1].xaxis.set_tick_params(labelbottom=False)
# axs[0,2].xaxis.set_tick_params(labelbottom=False)
# axs[1,2].xaxis.set_tick_params(labelbottom=False)

axs[0,0].set_title('Sensor 1')
axs[0,1].set_title('Sensor 2')
axs[0,2].set_title('Sensor 3')

# bias_h1 = np.mean(dataset_r_h[:,:,sensors[0]]*100 - full_res_r_h[:,:,sensors[0]]*100)
# bias_u1 = np.mean(dataset_r_u[:,:,sensors[0]]*100 - full_res_r_u[:,:,sensors[0]]*100)
# bias_v1 = np.mean(dataset_r_v[:,:,sensors[0]]*100 - full_res_r_v[:,:,sensors[0]]*100)

# bias_h2 = np.mean(dataset_r_h[:,:,sensors[1]]*100 - full_res_r_h[:,:,sensors[1]]*100)
# bias_u2 = np.mean(dataset_r_u[:,:,sensors[1]]*100 - full_res_r_u[:,:,sensors[1]]*100)
# bias_v2 = np.mean(dataset_r_v[:,:,sensors[1]]*100 - full_res_r_v[:,:,sensors[1]]*100)

# bias_h3 = np.mean(dataset_r_h[:,:,sensors[2]]*100 - full_res_r_h[:,:,sensors[2]]*100)
# bias_u3 = np.mean(dataset_r_u[:,:,sensors[2]]*100 - full_res_r_u[:,:,sensors[2]]*100)
# bias_v3 = np.mean(dataset_r_v[:,:,sensors[2]]*100 - full_res_r_v[:,:,sensors[2]]*100)


def panel(ax, x_pt, y_pt):
    # x_pt, y_pt: arrays (P, T, S)
    x = (x_pt.reshape(P, T, -1)[:, :, s] * 100.0).reshape(-1)  # (P*T,)
    y = (y_pt.reshape(P, T, -1)[:, :, s] * 100.0).reshape(-1)  # (P*T,)
    ax.scatter(x, y, s=5, c=c_all, cmap=cmap, norm=norm)
    lo = np.nanmin([x.min(), y.min()])
    hi = np.nanmax([x.max(), y.max()])
    ax.plot([lo, hi], [lo, hi], '--', color='k', linewidth=1)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect('equal', adjustable='box')

for j in range(3):
    s = sensors[j]

    panel(axs[0, j], dataset_r_h, full_res_r_h)

    panel(axs[1, j], dataset_r_u, full_res_r_u)

    panel(axs[2, j], dataset_r_v, full_res_r_v)

sm = cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axs, fraction=0.025, pad=0.02)
cbar.set_label('Bottom Friction Coefficient')

# # Add a text box showing the extent for H, U, and V
# bias_text_h1 = f'Bias = {bias_h1:.2f}'
# bias_text_h2 = f'Bias = {bias_h2:.2f}'
# bias_text_h3 = f'Bias = {bias_h3:.2f}'
# bias_text_u1 = f'Bias = {bias_u1:.2f}'
# bias_text_u2 = f'Bias = {bias_u2:.2f}'
# bias_text_u3 = f'Bias = {bias_u3:.2f}'
# bias_text_v1 = f'Bias = {bias_v1:.2f}'
# bias_text_v2 = f'Bias = {bias_v2:.2f}'
# bias_text_v3 = f'Bias = {bias_v3:.2f}'

# axs[0,0].text(0.5, 0.95, bias_text_h1, transform=axs[0,0].transAxes, 
#               fontsize=15, verticalalignment='top', horizontalalignment='right',
#               bbox=dict(facecolor='white', alpha=0.5))
# axs[0,1].text(0.5, 0.95, bias_text_h2, transform=axs[0,1].transAxes, 
#               fontsize=15, verticalalignment='top', horizontalalignment='right',
#               bbox=dict(facecolor='white', alpha=0.5))
# axs[0,2].text(0.5, 0.95, bias_text_h3, transform=axs[0,2].transAxes, 
#               fontsize=15, verticalalignment='top', horizontalalignment='right',
#               bbox=dict(facecolor='white', alpha=0.5))
# axs[1,0].text(0.5, 0.95, bias_text_u1, transform=axs[1,0].transAxes, 
#               fontsize=15, verticalalignment='top', horizontalalignment='right',
#               bbox=dict(facecolor='white', alpha=0.5))
# axs[1,1].text(0.5, 0.95, bias_text_u2, transform=axs[1,1].transAxes, 
#               fontsize=15, verticalalignment='top', horizontalalignment='right',
#               bbox=dict(facecolor='white', alpha=0.5))
# axs[1,2].text(0.5, 0.95, bias_text_u3, transform=axs[1,2].transAxes, 
#               fontsize=15, verticalalignment='top', horizontalalignment='right',
#               bbox=dict(facecolor='white', alpha=0.5))
# axs[2,0].text(0.5, 0.95, bias_text_v1, transform=axs[2,0].transAxes, 
#               fontsize=15, verticalalignment='top', horizontalalignment='right',
#               bbox=dict(facecolor='white', alpha=0.5))
# axs[2,1].text(0.5, 0.95, bias_text_v2, transform=axs[2,1].transAxes, 
#               fontsize=15, verticalalignment='top', horizontalalignment='right',
#               bbox=dict(facecolor='white', alpha=0.5))
# axs[2,2].text(0.5, 0.95, bias_text_v3, transform=axs[2,2].transAxes, 
#               fontsize=15, verticalalignment='top', horizontalalignment='right',
#               bbox=dict(facecolor='white', alpha=0.5))

plt.show()



#### ACC and RMSE Table

In [ ]:
##### ACC
P = dataset_r_h.shape[0]
MM = dataset_r_h.shape[1]
N = dataset_r_h.shape[2]

ACC_h = np.zeros(dataset_r_h.shape[0])

for p in range(P):
    num = np.zeros(MM)
    den = np.zeros(MM)
    for j in range(MM):
        Pj = np.sum(full_res_r_h[p,j,:]*100)/N
        Tj = np.sum(dataset_r_h[p,j,:]*100)/N
        num[j] = np.sum((dataset_r_h[p,j,:]*100-Tj)*(full_res_r_h[p,j,:]*100-Pj))
        den[j] = np.sqrt(np.sum((dataset_r_h[p,j,:]*100-Tj)**2)*np.sum((full_res_r_h[p,j,:]*100-Pj)**2))
    ACC_h[p] = np.sum(num/den)/MM
        # for i in range(dataset_r_h.shape[2]):
            
print('ACC:'+str(list(ACC_h)))
print('RMSE:'+str(list(np.mean(rmse_h,axis=1))))
print('nRMSE:'+str(list(np.mean(n_rmse_h,axis=1))))
print('uRMSE:'+str(list(np.mean(ub_s_rmse_h,axis=1))))
print('Max RMSE:'+str(list(np.max(rmse_h,axis=1))))


In [ ]:
##### ACC

P = dataset_r_u.shape[0]
MM = dataset_r_u.shape[1]
N = dataset_r_u.shape[2]

ACC_u = np.zeros(dataset_r_u.shape[0])

for p in range(P):
    num = np.zeros(MM)
    den = np.zeros(MM)
    for j in range(MM):
        Pj = np.sum(full_res_r_u[p,j,:]*100)/N
        Tj = np.sum(dataset_r_u[p,j,:]*100)/N
        num[j] = np.sum((dataset_r_u[p,j,:]*100-Tj)*(full_res_r_u[p,j,:]*100-Pj))
        den[j] = np.sqrt(np.sum((dataset_r_u[p,j,:]*100-Tj)**2)*np.sum((full_res_r_u[p,j,:]*100-Pj)**2))
    ACC_u[p] = np.sum(num/den)/MM
        # for i in range(dataset_r_u.shape[2]):
            
print('ACC:'+str(list(ACC_u)))
print('RMSE:'+str(list(np.mean(rmse_u,axis=1))))
print('nRMSE:'+str(list(np.mean(n_rmse_u,axis=1))))
print('uRMSE:'+str(list(np.mean(ub_s_rmse_u,axis=1))))
print('Max RMSE:'+str(list(np.max(rmse_u,axis=1))))


In [ ]:
##### ACC

P = dataset_r_v.shape[0]
MM = dataset_r_v.shape[1]
N = dataset_r_v.shape[2]

ACC_v = np.zeros(dataset_r_v.shape[0])

for p in range(P):
    num = np.zeros(MM)
    den = np.zeros(MM)
    for j in range(MM):
        Pj = np.sum(full_res_r_v[p,j,:]*100)/N
        Tj = np.sum(dataset_r_v[p,j,:]*100)/N
        num[j] = np.sum((dataset_r_v[p,j,:]*100-Tj)*(full_res_r_v[p,j,:]*100-Pj))
        den[j] = np.sqrt(np.sum((dataset_r_v[p,j,:]*100-Tj)**2)*np.sum((full_res_r_v[p,j,:]*100-Pj)**2))
    ACC_v[p] = np.sum(num/den)/MM
        # for i in range(dataset_r_v.shape[2]):
            
print('ACC:'+str(list(ACC_v)))
print('RMSE:'+str(list(np.mean(rmse_v,axis=1))))
print('nRMSE:'+str(list(np.mean(n_rmse_v,axis=1))))
print('uRMSE:'+str(list(np.mean(ub_s_rmse_v,axis=1))))
print('Max RMSE:'+str(list(np.max(rmse_v,axis=1))))


### Predictions 2

In [ ]:
window_size = 5
day_window = 5.
day_total = test_days[1]-test_days[0]

n_windows = int(day_total/day_window)

snap_windows = int(Nt_snaps/day_total*day_window)

rmse_array_h = np.zeros((n_windows, len(param_list_test)))
rmse_array_u = np.zeros((n_windows, len(param_list_test)))
rmse_array_v = np.zeros((n_windows, len(param_list_test)))

max_array_h = np.zeros((n_windows, len(param_list_test)))
max_array_u = np.zeros((n_windows, len(param_list_test)))
max_array_v = np.zeros((n_windows, len(param_list_test)))

In [ ]:
current_window = 0

for w in tqdm(range(n_windows)):
    res_h = np.zeros((Np, snap_windows, latent_dim_h))
    res_u = np.zeros((Np, snap_windows, latent_dim_u))
    res_v = np.zeros((Np, snap_windows, latent_dim_v))
    
    for p in range(Np):

        b_ic_h = ls_scaler_h.transform(np.expand_dims(data_ls_h[p, current_window, :],0))[0]#b_ic_data_h[p, current_window,:].copy()#
        res_h[p,0,:] = b_ic_h
        
        b_ic_u = ls_scaler_u.transform(np.expand_dims(data_ls_u[p, current_window, :],0))[0]
        res_u[p,0,:] = b_ic_u
        
        b_ic_v = ls_scaler_v.transform(np.expand_dims(data_ls_v[p, current_window, :],0))[0]
        res_v[p,0,:] = b_ic_v
        
        for i in range(0, snap_windows, window_size):
            temp_i = i + current_window

            if i+window_size+1 < snap_windows:
                
                par_input_h = b_par_data_h[p, temp_i+1:temp_i+window_size+1, :]   
                ic_input_h = np.tile(np.expand_dims(b_ic_h,1),window_size).T 
                bc_input_h = b_bc_data_h[p, temp_i+1:temp_i+window_size+1, :]  
                t_input_h = np.expand_dims(t_data_h[0:window_size],1)  
                prediction_h = mito_model_h([par_input_h, ic_input_h, bc_input_h, t_input_h]).numpy()
                res_h[p,i+1:i+window_size+1] = prediction_h 
                b_ic_h = prediction_h[-1].copy()   # Update initial condition for the next iteration

                par_input_u = b_par_data_u[p, temp_i+1:temp_i+window_size+1, :]    
                ic_input_u = np.tile(np.expand_dims(b_ic_u,1),window_size).T 
                bc_input_u = b_bc_data_u[p, temp_i+1:temp_i+window_size+1, :]   
                t_input_u = np.expand_dims(t_data_u[0:window_size],1)  
                prediction_u = mito_model_u([par_input_u, ic_input_u, bc_input_u, t_input_u]).numpy()
                res_u[p,i+1:i+window_size+1] = prediction_u 
                b_ic_u = prediction_u[-1]   # Update initial condition for the next iteration

                par_input_v = b_par_data_v[p, temp_i+1:temp_i+window_size+1, :]    
                ic_input_v = np.tile(np.expand_dims(b_ic_v,1),window_size).T 
                bc_input_v = b_bc_data_v[p, temp_i+1:temp_i+window_size+1, :]   
                t_input_v = np.expand_dims(t_data_v[0:window_size],1)  
                prediction_v = mito_model_v([par_input_v, ic_input_v, bc_input_v, t_input_v]).numpy()
                res_v[p,i+1:i+window_size+1] = prediction_v 
                b_ic_v = prediction_v[-1]   # Update initial condition for the next iteration
            else:
                diff = snap_windows - i - 1
                
                par_input_h = b_par_data_h[p, temp_i+1:temp_i+1+diff, :]    
                ic_input_h = np.tile(np.expand_dims(b_ic_h,1),diff).T  
                bc_input_h = b_bc_data_h[p, temp_i+1:temp_i+1+diff, :]   
                t_input_h = np.expand_dims(t_data_h[0:diff],1)  
                prediction_h = mito_model_h([par_input_h, ic_input_h, bc_input_h, t_input_h]).numpy()
                res_h[p,i+1:i+diff+1] = prediction_h 
                
                par_input_u = b_par_data_u[p, temp_i+1:temp_i+1+diff, :]    
                ic_input_u = np.tile(np.expand_dims(b_ic_u,1),diff).T 
                bc_input_u = b_bc_data_u[p, temp_i+1:temp_i+1+diff, :]   
                t_input_u = np.expand_dims(t_data_u[0:diff],1)  
                prediction_u = mito_model_u([par_input_u, ic_input_u, bc_input_u, t_input_u]).numpy()
                res_u[p,i+1:i+diff+1] = prediction_u 
                
                par_input_v = b_par_data_v[p, temp_i+1:temp_i+1+diff, :]    
                ic_input_v = np.tile(np.expand_dims(b_ic_v,1),diff).T 
                bc_input_v = b_bc_data_v[p, temp_i+1:temp_i+1+diff, :]   
                t_input_v = np.expand_dims(t_data_v[0:diff],1)  
                prediction_v = mito_model_v([par_input_v, ic_input_v, bc_input_v, t_input_v]).numpy()
                res_v[p,i+1:i+diff+1] = prediction_v 

    res_h = ls_scaler_h.inverse_transform(res_h.reshape(Np*snap_windows,latent_dim_h))
    # res_h = ls_scaler_h.inverse_transform(res_h)        
    full_res_h = ae_model_h.decoder(res_h).numpy()
    full_res_h = np.squeeze(scaler_h.inverse_transform(full_res_h))
    full_res_r_h = np.reshape(full_res_h, (Np, snap_windows, Nn))

    res_u = ls_scaler_u.inverse_transform(res_u.reshape(Np*snap_windows,latent_dim_u))
    # res_u = ls_scaler_u.inverse_transform(res_u)        
    full_res_u = ae_model_u.decoder(res_u).numpy()
    full_res_u = np.squeeze(scaler_u.inverse_transform(full_res_u))
    full_res_r_u = np.reshape(full_res_u, (Np, snap_windows, Nn))

    res_v = ls_scaler_v.inverse_transform(res_v.reshape(Np*snap_windows,latent_dim_v))
    # res_v = ls_scaler_v.inverse_transform(res_v)        
    full_res_v = ae_model_v.decoder(res_v).numpy()
    full_res_v = np.squeeze(scaler_v.inverse_transform(full_res_v))
    full_res_r_v = np.reshape(full_res_v, (Np, snap_windows, Nn))

    
    for p in range(Np):

        rmse_array_h[w, p] = np.mean(np.sqrt(np.mean((full_res_r_h[p, :] - dataset_r_h[p, current_window:snap_windows+current_window] ) ** 2, axis=1)))*100
        max_array_h[w, p] = (np.abs(np.max(dataset_r_h[p, current_window:snap_windows+current_window])-np.min(dataset_r_h[p, current_window:snap_windows+current_window])))*100
        
        rmse_array_u[w, p] = np.mean(np.sqrt(np.mean((full_res_r_u[p, :] - dataset_r_u[p, current_window:snap_windows+current_window] ) ** 2, axis=1)))*100
        max_array_u[w, p] = (np.abs(np.max(dataset_r_u[p, current_window:snap_windows+current_window])-np.min(dataset_r_u[p, current_window:snap_windows+current_window])))*100
        
        rmse_array_v[w, p] = np.mean(np.sqrt(np.mean((full_res_r_v[p, :] - dataset_r_v[p, current_window:snap_windows+current_window] ) ** 2, axis=1)))*100
        max_array_v[w, p] = (np.abs(np.max(dataset_r_v[p, current_window:snap_windows+current_window])-np.min(dataset_r_v[p, current_window:snap_windows+current_window])))*100

    current_window += snap_windows



#### Plot5

In [ ]:
# colormap=[(0.8,0.8,0.8),(0.4,0.4,0.4),(0,0,0)]
colormap = [cmocean.cm.balance(0.375), cmocean.cm.balance(0.25), cmocean.cm.balance(0.125)]

days_list = ['5-10','10-15','15-20','20-25','25-30','30-35',
             '35-40','40-45','45-50','50-55','55-60']

# Create an array of positions for our x-axis
x = np.arange(len(days_list))

# Set the width of each bar
width = 0.25

fig, ax = plt.subplots(3,figsize=(15, 12), sharex=True, layout='constrained')

# We'll create one bar for each column in rmse_array_h
num_columns = 3

for p,i in enumerate([0,6,13]):
    # Offset each bar group by i*width
    offset = width * p
    
    # Plot the bars for column i
    rects = ax[0].bar(x + offset,
                   rmse_array_h[:, i]/max_array_h[:, i],
                   width,
                   color = colormap[p],
                   label=f'r = {param_list_test[i]}')
    
    # Plot the bars for column i
    rects = ax[1].bar(x + offset,
                   rmse_array_u[:, i]/max_array_u[:, i],
                   width,
                   color = colormap[p],
                   label=f'r = {param_list_test[i]}')
    
    # Plot the bars for column i
    rects = ax[2].bar(x + offset,
                   rmse_array_v[:, i]/max_array_v[:, i],
                   width,
                   color = colormap[p],
                   label=f'r = {param_list_test[i]}')
    
ax[0].set_ylim(0,0.009)
ax[1].set_ylim(0,0.016)
ax[2].set_ylim(0,0.014)

ax[0].set_ylabel('Water Surface Elevation')
ax[1].set_ylabel('U-Velocity')
ax[2].set_ylabel('V-Velocity')


# Adjust the x-axis ticks to be centered under the group
# We'll shift by width*(num_columns-1)/2 to center the labels
ax[2].set_xticks(x + width * (num_columns - 1) / 2)
ax[2].set_xticklabels(days_list)

ax[2].set_xlabel('Days')

ax[0].legend(bbox_to_anchor=(0.5,1.15), ncol=3, loc='center', title='Bottom Friction Coefficients', title_fontsize=15, fontsize=12.75)

# Calculate the min and max values for each dataset
min_h, max_h = np.min(rmse_array_h)*100, np.max(rmse_array_h)*100
min_u, max_u = np.min(rmse_array_u)*100, np.max(rmse_array_u)*100
min_v, max_v = np.min(rmse_array_v)*100, np.max(rmse_array_v)*100

# Add a text box showing the extent for H, U, and V
extent_text_h = f'{min_h:.2f} < $\overline{{RMSE}}$ (cm) < {max_h:.2f}'
extent_text_u = f'{min_u:.2f} < $\overline{{RMSE}}$ (cm/s) < {max_u:.2f}'
extent_text_v = f'{min_v:.2f} < $\overline{{RMSE}}$ (cm/s) < {max_v:.2f}'

# Add text box for Water Surface Elevation
ax[0].text(0.98, 0.95, extent_text_h, transform=ax[0].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

# Add text box for U-Velocity
ax[1].text(0.98, 0.95, extent_text_u, transform=ax[1].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

# Add text box for V-Velocity
ax[2].text(0.98, 0.95, extent_text_v, transform=ax[2].transAxes, 
              fontsize=15, verticalalignment='top', horizontalalignment='right',
              bbox=dict(facecolor='white', alpha=0.5))

fmt = '%.3f' # Format you want the ticks, e.g. '40%'
yticks = mtick.FormatStrFormatter(fmt)
ax[0].yaxis.set_major_formatter(yticks)
ax[1].yaxis.set_major_formatter(yticks)
ax[2].yaxis.set_major_formatter(yticks)



plt.show()

### Predictions 3

In [ ]:
window_size = 1

In [ ]:
if scaling:
    zero_h = scaler_h.transform(np.zeros((1,dataset_h.shape[1])))
    zero_u = scaler_u.transform(np.zeros((1,dataset_u.shape[1])))
    zero_v = scaler_v.transform(np.zeros((1,dataset_v.shape[1])))

    zero_ic_h = ls_scaler_h.transform(ae_model_h.encoder(zero_h))
    zero_ic_u = ls_scaler_u.transform(ae_model_u.encoder(zero_u))
    zero_ic_v = ls_scaler_v.transform(ae_model_v.encoder(zero_v))

In [ ]:
snaps_crop = [np.where(mesh['t']/60/60/24==45.)[0][0], np.where(mesh['t']/60/60/24==55.)[0][0]]
Nt_snaps_crop = mesh['t'][snaps_crop[0]:snaps_crop[1]].shape[0]
snaps_crop -= snaps[0]


In [ ]:
res_h = np.zeros((Np, Nt_snaps_crop, latent_dim_h))

for p in tqdm(range(Np)):
    b_ic = zero_ic_h[0] 
    res_h[p,0,:] = b_ic
    for i in range(snaps_crop[0],snaps_crop[1], window_size):
        if i+window_size < Nt_snaps:
            par_input = b_par_data_h[p, i+1:i+window_size+1, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
            bc_input = b_bc_data_h[p, i+1:i+window_size+1, :] 
            t_input = np.expand_dims(t_data_h[0:window_size],1)  
    
            prediction = mito_model_h([par_input, ic_input, bc_input, t_input])
            res_h[p,i+1-snaps_crop[0]:i+window_size+1-snaps_crop[0]] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
        else:
            diff = Nt_snaps - i - 1
            par_input = b_par_data_h[p, i+1:i+1+diff, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
            bc_input = b_bc_data_h[p, i+1:i+1+diff, :] 
            t_input = np.expand_dims(t_data_h[0:diff],1)  

            prediction = mito_model_h([par_input, ic_input, bc_input, t_input])
            res_h[p,i+1-snaps_crop[0]:i+diff+1-snaps_crop[0]] = prediction
            

res_h = ls_scaler_h.inverse_transform(res_h.reshape(Np*Nt_snaps_crop,latent_dim_h))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_h = ae_model_h.decoder(res_h).numpy()
full_res_h = np.squeeze(scaler_h.inverse_transform(full_res_h))

# Reshape data for analysis and visualization
full_res_r_h = np.reshape(full_res_h, (Np, Nt_snaps_crop, Nn))

In [ ]:
res_u = np.zeros((Np, Nt_snaps_crop, latent_dim_u))

for p in tqdm(range(Np)):
    b_ic = zero_ic_u[0] 
    res_u[p,0,:] = b_ic
    for i in range(snaps_crop[0],snaps_crop[1], window_size):
        if i+window_size < Nt_snaps:
            par_input = b_par_data_u[p, i+1:i+window_size+1, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
            bc_input = b_bc_data_u[p, i+1:i+window_size+1, :] 
            t_input = np.expand_dims(t_data_u[0:window_size],1)  
    
            prediction = mito_model_u([par_input, ic_input, bc_input, t_input])
            res_u[p,i+1-snaps_crop[0]:i+window_size+1-snaps_crop[0]] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
        else:
            diff = Nt_snaps - i - 1
            par_input = b_par_data_u[p, i+1:i+1+diff, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
            bc_input = b_bc_data_u[p, i+1:i+1+diff, :] 
            t_input = np.expand_dims(t_data_u[0:diff],1)  

            prediction = mito_model_u([par_input, ic_input, bc_input, t_input])
            res_u[p,i+1-snaps_crop[0]:i+diff+1-snaps_crop[0]] = prediction

res_u = ls_scaler_u.inverse_transform(res_u.reshape(Np*Nt_snaps_crop,latent_dim_u))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_u = ae_model_u.decoder(res_u).numpy()
full_res_u = np.squeeze(scaler_u.inverse_transform(full_res_u))

# Reshape data for analysis and visualization
full_res_r_u = np.reshape(full_res_u, (Np, Nt_snaps_crop, Nn))

In [ ]:
res_v = np.zeros((Np, Nt_snaps_crop, latent_dim_v))

for p in tqdm(range(Np)):
    b_ic = zero_ic_v[0] 
    res_v[p,0,:] = b_ic
    for i in range(snaps_crop[0],snaps_crop[1], window_size):
        if i+window_size < Nt_snaps:
            par_input = b_par_data_v[p, i+1:i+window_size+1, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
            bc_input = b_bc_data_v[p, i+1:i+window_size+1, :] 
            t_input = np.expand_dims(t_data_v[0:window_size],1)  
    
            prediction = mito_model_v([par_input, ic_input, bc_input, t_input])
            res_v[p,i+1-snaps_crop[0]:i+window_size+1-snaps_crop[0]] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
        else:
            diff = Nt_snaps - i - 1
            par_input = b_par_data_v[p, i+1:i+1+diff, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
            bc_input = b_bc_data_v[p, i+1:i+1+diff, :] 
            t_input = np.expand_dims(t_data_v[0:diff],1)  

            prediction = mito_model_v([par_input, ic_input, bc_input, t_input])
            res_v[p,i+1-snaps_crop[0]:i+diff+1-snaps_crop[0]] = prediction

res_v = ls_scaler_v.inverse_transform(res_v.reshape(Np*Nt_snaps_crop,latent_dim_v))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_v = ae_model_v.decoder(res_v).numpy()
full_res_v = np.squeeze(scaler_v.inverse_transform(full_res_v))

# Reshape data for analysis and visualization
full_res_r_v = np.reshape(full_res_v, (Np, Nt_snaps_crop, Nn))

#### Plot6

In [ ]:
x_lower = mesh['t'][snaps_crop[0]+snaps[0]]/3600/24 
x_upper = mesh['t'][snaps_crop[1]+snaps[0]]/3600/24
time_crop = mesh['t'][snaps_crop[0]+snaps[0]:snaps_crop[1]+snaps[0]]/60/60/24

fig2, axs2 = plt.subplots(3, 1, figsize=(15,11), sharex=True, layout='constrained')

axs2[0].plot(time_crop,full_res_r_h[6,:,sensors[0]]*100,label='MITONet (sensor 1)', linewidth=2, linestyle='-',color='blue')
axs2[0].plot(time_crop,dataset_r_h[6,snaps_crop[0]:snaps_crop[1],sensors[0]]*100,label='ADCIRC (sensor 1)', linewidth=2, linestyle='--',color='blue')
axs2[0].plot(time_crop,full_res_r_h[6,:,sensors[1]]*100,label='MITONet (sensor 2)', linewidth=2, linestyle='-',color='orange')
axs2[0].plot(time_crop,dataset_r_h[6,snaps_crop[0]:snaps_crop[1],sensors[1]]*100,label='ADCIRC (sensor 2)', linewidth=2, linestyle='--',color='orange')
axs2[0].plot(time_crop,full_res_r_h[6,:,sensors[2]]*100,label='MITONet (sensor 3)', linewidth=2, linestyle='-',color='green')
axs2[0].plot(time_crop,dataset_r_h[6,snaps_crop[0]:snaps_crop[1],sensors[2]]*100,label='ADCIRC (sensor 3)', linewidth=2, linestyle='--',color='green')
axs2[0].set_xlim(x_lower,x_upper)
axs2[0].set_ylabel('Water Surface Elevation (cm)')
axs2[0].legend(bbox_to_anchor=(0.5,1.2), loc='upper center', ncol = 6, columnspacing=0.5, fontsize=12)

axs2[1].plot(time_crop,full_res_r_u[6,:,sensors[0]]*100,label='MITONet (sensor 1)', linewidth=2, linestyle='-',color='blue')
axs2[1].plot(time_crop,dataset_r_u[6,snaps_crop[0]:snaps_crop[1],sensors[0]]*100,label='ADCIRC (sensor 1)', linewidth=2, linestyle='--',color='blue')
axs2[1].plot(time_crop,full_res_r_u[6,:,sensors[1]]*100,label='MITONet (sensor 2)', linewidth=2, linestyle='-',color='orange')
axs2[1].plot(time_crop,dataset_r_u[6,snaps_crop[0]:snaps_crop[1],sensors[1]]*100,label='ADCIRC (sensor 2)', linewidth=2, linestyle='--',color='orange')
axs2[1].plot(time_crop,full_res_r_u[6,:,sensors[2]]*100,label='MITONet (sensor 3)', linewidth=2, linestyle='-',color='green')
axs2[1].plot(time_crop,dataset_r_u[6,snaps_crop[0]:snaps_crop[1],sensors[2]]*100,label='ADCIRC (sensor 3)', linewidth=2, linestyle='--',color='green')
axs2[1].set_xlim(x_lower,x_upper)
axs2[1].set_ylabel('U-Velocity (cm/s)')

axs2[2].plot(time_crop,full_res_r_v[6,:,sensors[0]]*100,label='MITONet (sensor 1)', linewidth=2, linestyle='-',color='blue')
axs2[2].plot(time_crop,dataset_r_v[6,snaps_crop[0]:snaps_crop[1],sensors[0]]*100,label='ADCIRC (sensor 1)', linewidth=2, linestyle='--',color='blue')
axs2[2].plot(time_crop,full_res_r_v[6,:,sensors[1]]*100,label='MITONet (sensor 2)', linewidth=2, linestyle='-',color='orange')
axs2[2].plot(time_crop,dataset_r_v[6,snaps_crop[0]:snaps_crop[1],sensors[1]]*100,label='ADCIRC (sensor 2)', linewidth=2, linestyle='--',color='orange')
axs2[2].plot(time_crop,full_res_r_v[6,:,sensors[2]]*100,label='MITONet (sensor 3)', linewidth=2, linestyle='-',color='green')
axs2[2].plot(time_crop,dataset_r_v[6,snaps_crop[0]:snaps_crop[1],sensors[2]]*100,label='ADCIRC (sensor 3)', linewidth=2, linestyle='--',color='green')
axs2[2].set_xlim(x_lower,x_upper)
axs2[2].set_ylabel('V-Velocity (cm/s)')
axs2[2].set_xlabel('Time (days)')
axs2[2].set_xticks(np.arange(time[snaps_crop[0]],time[snaps_crop[1]]+1,1))


In [ ]:
tstep1 = 0
tstep2 = tstep1 + int((0.25*24)*60/30)
tstep3 = tstep1 + int((1*24)*60/30)
tstep4 = tstep1 + int((5*24)*60/30)
param = 6

max_h = np.max((full_res_r_h[param,tstep1,:],full_res_r_h[param,tstep2,:],full_res_r_h[param,tstep3,:],full_res_r_h[param,tstep4,:]))*100
min_h = np.min((full_res_r_h[param,tstep1,:],full_res_r_h[param,tstep2,:],full_res_r_h[param,tstep3,:],full_res_r_h[param,tstep4,:]))*100
max_u = np.max((full_res_r_u[param,tstep1,:],full_res_r_u[param,tstep2,:],full_res_r_u[param,tstep3,:],full_res_r_u[param,tstep4,:]))*100
min_u = np.min((full_res_r_u[param,tstep1,:],full_res_r_u[param,tstep2,:],full_res_r_u[param,tstep3,:],full_res_r_u[param,tstep4,:]))*100
max_v = np.max((full_res_r_v[param,tstep1,:],full_res_r_v[param,tstep2,:],full_res_r_v[param,tstep3,:],full_res_r_v[param,tstep4,:]))*100
min_v = np.min((full_res_r_v[param,tstep1,:],full_res_r_v[param,tstep2,:],full_res_r_v[param,tstep3,:],full_res_r_v[param,tstep4,:]))*100


fig, axs = plt.subplots(3, 4, figsize=(20, 10), layout='constrained')#, sharey='col')

axs[0,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_h[param,tstep1,:]*100, vmin=min_h, 
                   vmax=max_h, cmap='cmo.delta', shading='gouraud')
axs[0,0].axis('off')

axs[0,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_h[param,tstep2,:]*100, vmin=min_h, 
                   vmax=max_h, cmap='cmo.delta', shading='gouraud')
axs[0,1].axis('off')

axs[0,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_h[param,tstep3,:]*100, vmin=min_h, 
                   vmax=max_h, cmap='cmo.delta', shading='gouraud')
axs[0,2].axis('off')

last_h = axs[0,3].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                           full_res_r_h[param,tstep4,:]*100, vmin=min_h, 
                           vmax=max_h, cmap='cmo.delta', shading='gouraud')
axs[0,3].axis('off')


axs[1,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_u[param,tstep1,:]*100, vmin=min_u, 
                   vmax=max_u, cmap='cmo.delta', shading='gouraud')
axs[1,0].axis('off')
axs[1,0].set_xlim([-72.50, -72.46])
axs[1,0].set_ylim([40.82,40.88]) 

axs[1,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_u[param,tstep2,:]*100, vmin=min_u, 
                   vmax=max_u, cmap='cmo.delta', shading='gouraud')
axs[1,1].axis('off')
axs[1,1].set_xlim([-72.50, -72.46])
axs[1,1].set_ylim([40.82,40.88]) 

axs[1,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_u[param,tstep3,:]*100, vmin=min_u, 
                   vmax=max_u, cmap='cmo.delta', shading='gouraud')
axs[1,2].axis('off')
axs[1,2].set_xlim([-72.50, -72.46])
axs[1,2].set_ylim([40.82,40.88]) 

last_u = axs[1,3].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                           full_res_r_u[param,tstep4,:]*100, vmin=min_u, 
                           vmax=max_u, cmap='cmo.delta', shading='gouraud')
axs[1,3].axis('off')
axs[1,3].set_xlim([-72.50, -72.46])
axs[1,3].set_ylim([40.82,40.88]) 

axs[2,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_v[param,tstep1,:]*100, vmin=min_v, 
                   vmax=max_v, cmap='cmo.delta', shading='gouraud')
axs[2,0].axis('off')
axs[2,0].set_xlim([-72.50, -72.46])
axs[2,0].set_ylim([40.82,40.88]) 

axs[2,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_v[param,tstep2,:]*100, vmin=min_v, 
                   vmax=max_v, cmap='cmo.delta', shading='gouraud')
axs[2,1].axis('off')
axs[2,1].set_xlim([-72.50, -72.46])
axs[2,1].set_ylim([40.82,40.88]) 

axs[2,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_v[param,tstep3,:]*100, vmin=min_v, 
                   vmax=max_v, cmap='cmo.delta', shading='gouraud')
axs[2,2].axis('off')
axs[2,2].set_xlim([-72.50, -72.46])
axs[2,2].set_ylim([40.82,40.88]) 

last_v = axs[2,3].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                           full_res_r_v[param,tstep4,:]*100, vmin=min_v, 
                           vmax=max_v, cmap='cmo.delta', shading='gouraud')
axs[2,3].axis('off')
axs[2,3].set_xlim([-72.50, -72.46])
axs[2,3].set_ylim([40.82,40.88]) 

axs[0, 0].text(0.5, 1, 'Day {0}'.format(time[snaps_crop[0]+tstep1]), size=20, ha="center", va="center", 
               rotation=0, transform=axs[0,0].transAxes)
axs[0, 1].text(0.5, 1, 'Day {0}'.format(time[snaps_crop[0]+tstep2]), size=20, ha="center", va="center", 
               rotation=0, transform=axs[0,1].transAxes)
axs[0, 2].text(0.5, 1, 'Day {0}'.format(time[snaps_crop[0]+tstep3]), size=20, ha="center", va="center", 
               rotation=0, transform=axs[0,2].transAxes)
axs[0, 3].text(0.5, 1, 'Day {0}'.format(time[snaps_crop[0]+tstep4]), size=20, ha="center", va="center", 
               rotation=0, transform=axs[0,3].transAxes)

axs[0, 0].text(-0.1, 0.5, 'Water Surface Elevation', size=20, ha="center", va="center", 
               rotation=90, transform=axs[0,0].transAxes)
axs[1, 0].text(-0.1, 0.5, 'U-Velocity', size=20, ha="center", va="center", 
               rotation=90, transform=axs[1,0].transAxes)
axs[2, 0].text(-0.1, 0.5, 'V-Velocity', size=20, ha="center", va="center", 
               rotation=90, transform=axs[2,0].transAxes)

cbar = plt.colorbar(last_h, orientation='vertical', aspect=15)
cbar.set_label(label='(cm)', size=20) 
cbar = plt.colorbar(last_u, orientation='vertical', aspect=15)
cbar.set_label(label='(cm/s)', size=20) 
cbar = plt.colorbar(last_v, orientation='vertical', aspect=15)
cbar.set_label(label='(cm/s)',  size=20) 

rmse_h1 = np.sqrt(np.mean((full_res_r_h[param, tstep1, mask_num:] - dataset_r_h[param, snaps_crop[0]+tstep1, mask_num:]) ** 2))*100
rmse_h1_text = f'$RMSE$ = {rmse_h1:.2f} cm'
rmse_h2 = np.sqrt(np.mean((full_res_r_h[param, tstep2, mask_num:] - dataset_r_h[param, snaps_crop[0]+tstep2, mask_num:]) ** 2))*100
rmse_h2_text = f'$RMSE$ = {rmse_h2:.2f} cm'
rmse_h3 = np.sqrt(np.mean((full_res_r_h[param, tstep3, mask_num:] - dataset_r_h[param, snaps_crop[0]+tstep3, mask_num:]) ** 2))*100
rmse_h3_text = f'$RMSE$ = {rmse_h3:.2f} cm'
rmse_h4 = np.sqrt(np.mean((full_res_r_h[param, tstep4, mask_num:] - dataset_r_h[param, snaps_crop[0]+tstep4, mask_num:]) ** 2))*100
rmse_h4_text = f'$RMSE$ = {rmse_h4:.2f} cm'

rmse_u1 = np.sqrt(np.mean((full_res_r_u[param, tstep1, mask_num:] - dataset_r_u[param, snaps_crop[0]+tstep1, mask_num:]) ** 2))*100
rmse_u1_text = f'$RMSE$ = {rmse_u1:.2f} cm/s'
rmse_u2 = np.sqrt(np.mean((full_res_r_u[param, tstep2, mask_num:] - dataset_r_u[param, snaps_crop[0]+tstep2, mask_num:]) ** 2))*100
rmse_u2_text = f'$RMSE$ = {rmse_u2:.2f} cm/s'
rmse_u3 = np.sqrt(np.mean((full_res_r_u[param, tstep3, mask_num:] - dataset_r_u[param, snaps_crop[0]+tstep3, mask_num:]) ** 2))*100
rmse_u3_text = f'$RMSE$ = {rmse_u3:.2f} cm/s'
rmse_u4 = np.sqrt(np.mean((full_res_r_u[param, tstep4, mask_num:] - dataset_r_u[param, snaps_crop[0]+tstep4, mask_num:]) ** 2))*100
rmse_u4_text = f'$RMSE$ = {rmse_u4:.2f} cm/s'

rmse_v1 = np.sqrt(np.mean((full_res_r_v[param, tstep1, mask_num:] - dataset_r_v[param, snaps_crop[0]+tstep1, mask_num:]) ** 2))*100
rmse_v1_text = f'$RMSE$ = {rmse_v1:.2f} cm/s'
rmse_v2 = np.sqrt(np.mean((full_res_r_v[param, tstep2, mask_num:] - dataset_r_v[param, snaps_crop[0]+tstep2, mask_num:]) ** 2))*100
rmse_v2_text = f'$RMSE$ = {rmse_v2:.2f} cm/s'
rmse_v3 = np.sqrt(np.mean((full_res_r_v[param, tstep3, mask_num:] - dataset_r_v[param, snaps_crop[0]+tstep3, mask_num:]) ** 2))*100
rmse_v3_text = f'$RMSE$ = {rmse_v3:.2f} cm/s'
rmse_v4 = np.sqrt(np.mean((full_res_r_v[param, tstep4, mask_num:] - dataset_r_v[param, snaps_crop[0]+tstep4, mask_num:]) ** 2))*100
rmse_v4_text = f'$RMSE$ = {rmse_v4:.2f} cm/s'

axs[0,0].text(0.5, -0.05, rmse_h1_text, transform=axs[0,0].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[0,1].text(0.5, -0.05, rmse_h2_text, transform=axs[0,1].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[0,2].text(0.5, -0.05, rmse_h3_text, transform=axs[0,2].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[0,3].text(0.5, -0.05, rmse_h4_text, transform=axs[0,3].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')

axs[1,0].text(0.5, 0.0, rmse_u1_text, transform=axs[1,0].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[1,1].text(0.5, 0.0, rmse_u2_text, transform=axs[1,1].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[1,2].text(0.5, 0.0, rmse_u3_text, transform=axs[1,2].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[1,3].text(0.5, 0.0, rmse_u4_text, transform=axs[1,3].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')

axs[2,0].text(0.5, 0.0, rmse_v1_text, transform=axs[2,0].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[2,1].text(0.5, 0.0, rmse_v2_text, transform=axs[2,1].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[2,2].text(0.5, 0.0, rmse_v3_text, transform=axs[2,2].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[2,3].text(0.5, 0.0, rmse_v4_text, transform=axs[2,3].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')

### Predictions 4

In [ ]:
res_h = np.zeros((Np, Nt_snaps_crop, latent_dim_h))

for p in tqdm(range(Np)):
    b_ic = ls_scaler_h.transform(np.expand_dims(data_ls_h[p, snaps_crop[0], :],0))[0]
    res_h[p,0,:] = b_ic
    for i in range(snaps_crop[0],snaps_crop[1], window_size):
        if i+window_size < Nt_snaps:
            par_input = b_par_data_h[p, i+1:i+window_size+1, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
            bc_input = b_bc_data_h[p, i+1:i+window_size+1, :] 
            t_input = np.expand_dims(t_data_h[0:window_size],1)  
    
            prediction = mito_model_h([par_input, ic_input, bc_input, t_input])
            res_h[p,i+1-snaps_crop[0]:i+window_size+1-snaps_crop[0]] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
        else:
            diff = Nt_snaps - i - 1
            par_input = b_par_data_h[p, i+1:i+1+diff, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
            bc_input = b_bc_data_h[p, i+1:i+1+diff, :] 
            t_input = np.expand_dims(t_data_h[0:diff],1)  

            prediction = mito_model_h([par_input, ic_input, bc_input, t_input])
            res_h[p,i+1-snaps_crop[0]:i+diff+1-snaps_crop[0]] = prediction
            

res_h = ls_scaler_h.inverse_transform(res_h.reshape(Np*Nt_snaps_crop,latent_dim_h))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_h = ae_model_h.decoder(res_h).numpy()
full_res_h = np.squeeze(scaler_h.inverse_transform(full_res_h))

# Reshape data for analysis and visualization
full_res_r_h = np.reshape(full_res_h, (Np, Nt_snaps_crop, Nn))

In [ ]:
res_u = np.zeros((Np, Nt_snaps_crop, latent_dim_u))

for p in tqdm(range(Np)):
    b_ic = ls_scaler_u.transform(np.expand_dims(data_ls_u[p, snaps_crop[0], :],0))[0]
    res_u[p,0,:] = b_ic
    for i in range(snaps_crop[0],snaps_crop[1], window_size):
        if i+window_size < Nt_snaps:
            par_input = b_par_data_u[p, i+1:i+window_size+1, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
            bc_input = b_bc_data_u[p, i+1:i+window_size+1, :] 
            t_input = np.expand_dims(t_data_u[0:window_size],1)  
    
            prediction = mito_model_u([par_input, ic_input, bc_input, t_input])
            res_u[p,i+1-snaps_crop[0]:i+window_size+1-snaps_crop[0]] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
        else:
            diff = Nt_snaps - i - 1
            par_input = b_par_data_u[p, i+1:i+1+diff, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
            bc_input = b_bc_data_u[p, i+1:i+1+diff, :] 
            t_input = np.expand_dims(t_data_u[0:diff],1)  

            prediction = mito_model_u([par_input, ic_input, bc_input, t_input])
            res_u[p,i+1-snaps_crop[0]:i+diff+1-snaps_crop[0]] = prediction

res_u = ls_scaler_u.inverse_transform(res_u.reshape(Np*Nt_snaps_crop,latent_dim_u))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_u = ae_model_u.decoder(res_u).numpy()
full_res_u = np.squeeze(scaler_u.inverse_transform(full_res_u))

# Reshape data for analysis and visualization
full_res_r_u = np.reshape(full_res_u, (Np, Nt_snaps_crop, Nn))

In [ ]:
res_v = np.zeros((Np, Nt_snaps_crop, latent_dim_v))

for p in tqdm(range(Np)):
    b_ic = ls_scaler_v.transform(np.expand_dims(data_ls_v[p, snaps_crop[0], :],0))[0]
    res_v[p,0,:] = b_ic
    for i in range(snaps_crop[0],snaps_crop[1], window_size):
        if i+window_size < Nt_snaps:
            par_input = b_par_data_v[p, i+1:i+window_size+1, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),window_size).T
            bc_input = b_bc_data_v[p, i+1:i+window_size+1, :] 
            t_input = np.expand_dims(t_data_v[0:window_size],1)  
    
            prediction = mito_model_v([par_input, ic_input, bc_input, t_input])
            res_v[p,i+1-snaps_crop[0]:i+window_size+1-snaps_crop[0]] = prediction
            b_ic = prediction[-1]  # Update initial condition for the next iteration
        else:
            diff = Nt_snaps - i - 1
            par_input = b_par_data_v[p, i+1:i+1+diff, :]  
            ic_input = np.tile(np.expand_dims(b_ic,1),diff).T
            bc_input = b_bc_data_v[p, i+1:i+1+diff, :] 
            t_input = np.expand_dims(t_data_v[0:diff],1)  

            prediction = mito_model_v([par_input, ic_input, bc_input, t_input])
            res_v[p,i+1-snaps_crop[0]:i+diff+1-snaps_crop[0]] = prediction

res_v = ls_scaler_v.inverse_transform(res_v.reshape(Np*Nt_snaps_crop,latent_dim_v))

#### Post-processing ####

# Decode the latent representation to get the full resolution
full_res_v = ae_model_v.decoder(res_v).numpy()
full_res_v = np.squeeze(scaler_v.inverse_transform(full_res_v))

# Reshape data for analysis and visualization
full_res_r_v = np.reshape(full_res_v, (Np, Nt_snaps_crop, Nn))

#### Plot6

In [ ]:
x_lower = mesh['t'][snaps_crop[0]+snaps[0]]/3600/24 
x_upper = mesh['t'][snaps_crop[1]+snaps[0]]/3600/24
time_crop = mesh['t'][snaps_crop[0]+snaps[0]:snaps_crop[1]+snaps[0]]/60/60/24

fig2, axs2 = plt.subplots(3, 1, figsize=(15,11), sharex=True, layout='constrained')

axs2[0].plot(time_crop,full_res_r_h[6,:,sensors[0]]*100,label='MITONet (sensor 1)', linewidth=2, linestyle='-',color='blue')
axs2[0].plot(time_crop,dataset_r_h[6,snaps_crop[0]:snaps_crop[1],sensors[0]]*100,label='ADCIRC (sensor 1)', linewidth=2, linestyle='--',color='blue')
axs2[0].plot(time_crop,full_res_r_h[6,:,sensors[1]]*100,label='MITONet (sensor 2)', linewidth=2, linestyle='-',color='orange')
axs2[0].plot(time_crop,dataset_r_h[6,snaps_crop[0]:snaps_crop[1],sensors[1]]*100,label='ADCIRC (sensor 2)', linewidth=2, linestyle='--',color='orange')
axs2[0].plot(time_crop,full_res_r_h[6,:,sensors[2]]*100,label='MITONet (sensor 3)', linewidth=2, linestyle='-',color='green')
axs2[0].plot(time_crop,dataset_r_h[6,snaps_crop[0]:snaps_crop[1],sensors[2]]*100,label='ADCIRC (sensor 3)', linewidth=2, linestyle='--',color='green')
axs2[0].set_xlim(x_lower,x_upper)
axs2[0].set_ylabel('Water Surface Elevation (cm)')
axs2[0].legend(bbox_to_anchor=(0.5,1.2), loc='upper center', ncol = 6, columnspacing=0.5, fontsize=12)

axs2[1].plot(time_crop,full_res_r_u[6,:,sensors[0]]*100,label='MITONet (sensor 1)', linewidth=2, linestyle='-',color='blue')
axs2[1].plot(time_crop,dataset_r_u[6,snaps_crop[0]:snaps_crop[1],sensors[0]]*100,label='ADCIRC (sensor 1)', linewidth=2, linestyle='--',color='blue')
axs2[1].plot(time_crop,full_res_r_u[6,:,sensors[1]]*100,label='MITONet (sensor 2)', linewidth=2, linestyle='-',color='orange')
axs2[1].plot(time_crop,dataset_r_u[6,snaps_crop[0]:snaps_crop[1],sensors[1]]*100,label='ADCIRC (sensor 2)', linewidth=2, linestyle='--',color='orange')
axs2[1].plot(time_crop,full_res_r_u[6,:,sensors[2]]*100,label='MITONet (sensor 3)', linewidth=2, linestyle='-',color='green')
axs2[1].plot(time_crop,dataset_r_u[6,snaps_crop[0]:snaps_crop[1],sensors[2]]*100,label='ADCIRC (sensor 3)', linewidth=2, linestyle='--',color='green')
axs2[1].set_xlim(x_lower,x_upper)
axs2[1].set_ylabel('U-Velocity (cm/s)')

axs2[2].plot(time_crop,full_res_r_v[6,:,sensors[0]]*100,label='MITONet (sensor 1)', linewidth=2, linestyle='-',color='blue')
axs2[2].plot(time_crop,dataset_r_v[6,snaps_crop[0]:snaps_crop[1],sensors[0]]*100,label='ADCIRC (sensor 1)', linewidth=2, linestyle='--',color='blue')
axs2[2].plot(time_crop,full_res_r_v[6,:,sensors[1]]*100,label='MITONet (sensor 2)', linewidth=2, linestyle='-',color='orange')
axs2[2].plot(time_crop,dataset_r_v[6,snaps_crop[0]:snaps_crop[1],sensors[1]]*100,label='ADCIRC (sensor 2)', linewidth=2, linestyle='--',color='orange')
axs2[2].plot(time_crop,full_res_r_v[6,:,sensors[2]]*100,label='MITONet (sensor 3)', linewidth=2, linestyle='-',color='green')
axs2[2].plot(time_crop,dataset_r_v[6,snaps_crop[0]:snaps_crop[1],sensors[2]]*100,label='ADCIRC (sensor 3)', linewidth=2, linestyle='--',color='green')
axs2[2].set_xlim(x_lower,x_upper)
axs2[2].set_ylabel('V-Velocity (cm/s)')
axs2[2].set_xlabel('Time (days)')
axs2[2].set_xticks(np.arange(time[snaps_crop[0]],time[snaps_crop[1]]+1,1))


In [ ]:
tstep1 = 0
tstep2 = tstep1 + int((0.25*24)*60/30)
tstep3 = tstep1 + int((1*24)*60/30)
tstep4 = tstep1 + int((5*24)*60/30)
param = 6

max_h = np.max((full_res_r_h[param,tstep1,:],full_res_r_h[param,tstep2,:],full_res_r_h[param,tstep3,:],full_res_r_h[param,tstep4,:]))*100
min_h = np.min((full_res_r_h[param,tstep1,:],full_res_r_h[param,tstep2,:],full_res_r_h[param,tstep3,:],full_res_r_h[param,tstep4,:]))*100
max_u = np.max((full_res_r_u[param,tstep1,:],full_res_r_u[param,tstep2,:],full_res_r_u[param,tstep3,:],full_res_r_u[param,tstep4,:]))*100
min_u = np.min((full_res_r_u[param,tstep1,:],full_res_r_u[param,tstep2,:],full_res_r_u[param,tstep3,:],full_res_r_u[param,tstep4,:]))*100
max_v = np.max((full_res_r_v[param,tstep1,:],full_res_r_v[param,tstep2,:],full_res_r_v[param,tstep3,:],full_res_r_v[param,tstep4,:]))*100
min_v = np.min((full_res_r_v[param,tstep1,:],full_res_r_v[param,tstep2,:],full_res_r_v[param,tstep3,:],full_res_r_v[param,tstep4,:]))*100


fig, axs = plt.subplots(3, 4, figsize=(20, 10), layout='constrained')#, sharey='col')

axs[0,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_h[param,tstep1,:]*100, vmin=min_h, 
                   vmax=max_h, cmap='cmo.delta', shading='gouraud')
axs[0,0].axis('off')

axs[0,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_h[param,tstep2,:]*100, vmin=min_h, 
                   vmax=max_h, cmap='cmo.delta', shading='gouraud')
axs[0,1].axis('off')

axs[0,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_h[param,tstep3,:]*100, vmin=min_h, 
                   vmax=max_h, cmap='cmo.delta', shading='gouraud')
axs[0,2].axis('off')

last_h = axs[0,3].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                           full_res_r_h[param,tstep4,:]*100, vmin=min_h, 
                           vmax=max_h, cmap='cmo.delta', shading='gouraud')
axs[0,3].axis('off')


axs[1,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_u[param,tstep1,:]*100, vmin=min_u, 
                   vmax=max_u, cmap='cmo.delta', shading='gouraud')
axs[1,0].axis('off')
axs[1,0].set_xlim([-72.50, -72.46])
axs[1,0].set_ylim([40.82,40.88]) 

axs[1,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_u[param,tstep2,:]*100, vmin=min_u, 
                   vmax=max_u, cmap='cmo.delta', shading='gouraud')
axs[1,1].axis('off')
axs[1,1].set_xlim([-72.50, -72.46])
axs[1,1].set_ylim([40.82,40.88]) 

axs[1,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_u[param,tstep3,:]*100, vmin=min_u, 
                   vmax=max_u, cmap='cmo.delta', shading='gouraud')
axs[1,2].axis('off')
axs[1,2].set_xlim([-72.50, -72.46])
axs[1,2].set_ylim([40.82,40.88]) 

last_u = axs[1,3].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                           full_res_r_u[param,tstep4,:]*100, vmin=min_u, 
                           vmax=max_u, cmap='cmo.delta', shading='gouraud')
axs[1,3].axis('off')
axs[1,3].set_xlim([-72.50, -72.46])
axs[1,3].set_ylim([40.82,40.88]) 

axs[2,0].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_v[param,tstep1,:]*100, vmin=min_v, 
                   vmax=max_v, cmap='cmo.delta', shading='gouraud')
axs[2,0].axis('off')
axs[2,0].set_xlim([-72.50, -72.46])
axs[2,0].set_ylim([40.82,40.88]) 

axs[2,1].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_v[param,tstep2,:]*100, vmin=min_v, 
                   vmax=max_v, cmap='cmo.delta', shading='gouraud')
axs[2,1].axis('off')
axs[2,1].set_xlim([-72.50, -72.46])
axs[2,1].set_ylim([40.82,40.88]) 

axs[2,2].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                   full_res_r_v[param,tstep3,:]*100, vmin=min_v, 
                   vmax=max_v, cmap='cmo.delta', shading='gouraud')
axs[2,2].axis('off')
axs[2,2].set_xlim([-72.50, -72.46])
axs[2,2].set_ylim([40.82,40.88]) 

last_v = axs[2,3].tripcolor(mesh['nodes'][:,0], mesh['nodes'][:,1], mesh['triangles']-1, 
                           full_res_r_v[param,tstep4,:]*100, vmin=min_v, 
                           vmax=max_v, cmap='cmo.delta', shading='gouraud')
axs[2,3].axis('off')
axs[2,3].set_xlim([-72.50, -72.46])
axs[2,3].set_ylim([40.82,40.88]) 

axs[0, 0].text(0.5, 1, 'Day {0}'.format(time[snaps_crop[0]+tstep1]), size=20, ha="center", va="center", 
               rotation=0, transform=axs[0,0].transAxes)
axs[0, 1].text(0.5, 1, 'Day {0}'.format(time[snaps_crop[0]+tstep2]), size=20, ha="center", va="center", 
               rotation=0, transform=axs[0,1].transAxes)
axs[0, 2].text(0.5, 1, 'Day {0}'.format(time[snaps_crop[0]+tstep3]), size=20, ha="center", va="center", 
               rotation=0, transform=axs[0,2].transAxes)
axs[0, 3].text(0.5, 1, 'Day {0}'.format(time[snaps_crop[0]+tstep4]), size=20, ha="center", va="center", 
               rotation=0, transform=axs[0,3].transAxes)

axs[0, 0].text(-0.1, 0.5, 'Water Surface Elevation', size=20, ha="center", va="center", 
               rotation=90, transform=axs[0,0].transAxes)
axs[1, 0].text(-0.1, 0.5, 'U-Velocity', size=20, ha="center", va="center", 
               rotation=90, transform=axs[1,0].transAxes)
axs[2, 0].text(-0.1, 0.5, 'V-Velocity', size=20, ha="center", va="center", 
               rotation=90, transform=axs[2,0].transAxes)

cbar = plt.colorbar(last_h, orientation='vertical', aspect=15)
cbar.set_label(label='(cm)', size=20) 
cbar = plt.colorbar(last_u, orientation='vertical', aspect=15)
cbar.set_label(label='(cm/s)', size=20) 
cbar = plt.colorbar(last_v, orientation='vertical', aspect=15)
cbar.set_label(label='(cm/s)',  size=20) 

rmse_h1 = np.sqrt(np.mean((full_res_r_h[param, tstep1, mask_num:] - dataset_r_h[param, snaps_crop[0]+tstep1, mask_num:]) ** 2))*100
rmse_h1_text = f'$RMSE$ = {rmse_h1:.2f} cm'
rmse_h2 = np.sqrt(np.mean((full_res_r_h[param, tstep2, mask_num:] - dataset_r_h[param, snaps_crop[0]+tstep2, mask_num:]) ** 2))*100
rmse_h2_text = f'$RMSE$ = {rmse_h2:.2f} cm'
rmse_h3 = np.sqrt(np.mean((full_res_r_h[param, tstep3, mask_num:] - dataset_r_h[param, snaps_crop[0]+tstep3, mask_num:]) ** 2))*100
rmse_h3_text = f'$RMSE$ = {rmse_h3:.2f} cm'
rmse_h4 = np.sqrt(np.mean((full_res_r_h[param, tstep4, mask_num:] - dataset_r_h[param, snaps_crop[0]+tstep4, mask_num:]) ** 2))*100
rmse_h4_text = f'$RMSE$ = {rmse_h4:.2f} cm'

rmse_u1 = np.sqrt(np.mean((full_res_r_u[param, tstep1, mask_num:] - dataset_r_u[param, snaps_crop[0]+tstep1, mask_num:]) ** 2))*100
rmse_u1_text = f'$RMSE$ = {rmse_u1:.2f} cm/s'
rmse_u2 = np.sqrt(np.mean((full_res_r_u[param, tstep2, mask_num:] - dataset_r_u[param, snaps_crop[0]+tstep2, mask_num:]) ** 2))*100
rmse_u2_text = f'$RMSE$ = {rmse_u2:.2f} cm/s'
rmse_u3 = np.sqrt(np.mean((full_res_r_u[param, tstep3, mask_num:] - dataset_r_u[param, snaps_crop[0]+tstep3, mask_num:]) ** 2))*100
rmse_u3_text = f'$RMSE$ = {rmse_u3:.2f} cm/s'
rmse_u4 = np.sqrt(np.mean((full_res_r_u[param, tstep4, mask_num:] - dataset_r_u[param, snaps_crop[0]+tstep4, mask_num:]) ** 2))*100
rmse_u4_text = f'$RMSE$ = {rmse_u4:.2f} cm/s'

rmse_v1 = np.sqrt(np.mean((full_res_r_v[param, tstep1, mask_num:] - dataset_r_v[param, snaps_crop[0]+tstep1, mask_num:]) ** 2))*100
rmse_v1_text = f'$RMSE$ = {rmse_v1:.2f} cm/s'
rmse_v2 = np.sqrt(np.mean((full_res_r_v[param, tstep2, mask_num:] - dataset_r_v[param, snaps_crop[0]+tstep2, mask_num:]) ** 2))*100
rmse_v2_text = f'$RMSE$ = {rmse_v2:.2f} cm/s'
rmse_v3 = np.sqrt(np.mean((full_res_r_v[param, tstep3, mask_num:] - dataset_r_v[param, snaps_crop[0]+tstep3, mask_num:]) ** 2))*100
rmse_v3_text = f'$RMSE$ = {rmse_v3:.2f} cm/s'
rmse_v4 = np.sqrt(np.mean((full_res_r_v[param, tstep4, mask_num:] - dataset_r_v[param, snaps_crop[0]+tstep4, mask_num:]) ** 2))*100
rmse_v4_text = f'$RMSE$ = {rmse_v4:.2f} cm/s'

axs[0,0].text(0.5, -0.05, rmse_h1_text, transform=axs[0,0].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[0,1].text(0.5, -0.05, rmse_h2_text, transform=axs[0,1].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[0,2].text(0.5, -0.05, rmse_h3_text, transform=axs[0,2].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[0,3].text(0.5, -0.05, rmse_h4_text, transform=axs[0,3].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')

axs[1,0].text(0.5, 0.0, rmse_u1_text, transform=axs[1,0].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[1,1].text(0.5, 0.0, rmse_u2_text, transform=axs[1,1].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[1,2].text(0.5, 0.0, rmse_u3_text, transform=axs[1,2].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[1,3].text(0.5, 0.0, rmse_u4_text, transform=axs[1,3].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')

axs[2,0].text(0.5, 0.0, rmse_v1_text, transform=axs[2,0].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[2,1].text(0.5, 0.0, rmse_v2_text, transform=axs[2,1].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[2,2].text(0.5, 0.0, rmse_v3_text, transform=axs[2,2].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')
axs[2,3].text(0.5, 0.0, rmse_v4_text, transform=axs[2,3].transAxes, 
              fontsize=15, verticalalignment='bottom', horizontalalignment='center')